# 00 - Master notebook: the whole pipeline in one guided tour

The same cells, figures, gates and JSON records as notebooks 01-06, in the order
§11.2 requires, arranged to fit a single A100 session. What this notebook is
*for*: running the pipeline end to end once, on the real dataset, watching every
gate as it passes, and ending with the thesis table and the §11.3 never-cut
checklist.

**The arrangement it assumes** (set up before opening this notebook):

1. **The dataset was generated on another GPU and uploaded to the Modal
   Volume.** From the repo root on that machine:

       python -m src.data.generate --out data/datasets

   (2000/400/400 samples, ~11 GB, several GPU-hours) and then, once only:

       modal volume put fno-wave-inverse-data data/datasets/train.h5 datasets/train.h5
       modal volume put fno-wave-inverse-data data/datasets/val.h5   datasets/val.h5
       modal volume put fno-wave-inverse-data data/datasets/test.h5  datasets/test.h5

   Stage B re-solves 20 stored samples on *this* GPU and compares against the
   stored phasors, so an interrupted upload or a corrupted file is caught before
   anything trains on it. If you would rather not push 11 GB through your own
   connection, the headless equivalent writes the same files to the same paths:

       modal run modal_app.py::generate_all        # any GPU, e.g. FNO_GPU=L4

2. **Notebook settings** (the gear icon): kernel on **A100**, and the Volume
   `fno-wave-inverse-data` attached at mount path `/vol/fno-data` -- the first
   path `bootstrap.py` looks for. Every figure, record and checkpoint this
   notebook writes lands on that Volume, so nothing is lost when the session
   ends.

3. **The code comes from GitHub**: fill in `GITHUB_TOKEN` (a fine-grained,
   read-only PAT) and `GITHUB_REPO` in the first cell. The second cell clones
   the repo into the kernel (or finds it already mounted / walks up from the
   working directory when run locally or headless).

**Budget on one A100, at the defaults** (`MODE = "fit"`, `TRAIN_HOURS = 5.0`):

| stage | what runs | wall clock |
|---|---|---|
| A | solver checks 1-5, slow checks included | 20-40 min |
| B | verify the uploaded dataset: projection, re-solve spot check, loader | 10-15 min |
| C | train the `full` arm; epochs auto-fit to `TRAIN_HOURS` via a 2-epoch probe | ~5.2 h |
| D | forward evaluation on the test split | ~5 min |
| E | gradient check, one inversion, misfit landscape, 12 + 24 inversions | 20-40 min |
| F | RingCNN baseline, out-of-family FDTD, detector, transfer, thesis table | 30-45 min |

Total ~6.5-7.5 h: inside an 8 h session with margin. Two things are deliberately
*not* in that budget:

- the **`nophys` ablation arm** (doubles the training cost; run it headless
  later with `modal run modal_app.py::train_arm --arm nophys` -- the thesis
  table in stage F shows it as `-` until then), and
- the **full 300-epoch schedule** (8-16 h; `MODE = "full"`, or the same headless
  command). If the forward gates in stage D come out marginally above 5% rel-L2
  at the fitted epoch count, that is the budget talking, not the architecture:
  re-run stages D-F after the headless 300-epoch run finishes -- each stage
  reads the checkpoint from the Volume and skips anything already done.

**Knobs are top-level assignments**, so the headless launcher can flip them the
same way it flips `SMOKE`/`QUICK` in the other notebooks (note the shell
quoting on string values):

    modal run modal_app.py::run_notebook --name 00_master --flags "MODE=\"smoke\""

In [ ]:
# Fill these two in, then Run All.
GITHUB_TOKEN = ""                    # fine-grained PAT, Contents: read-only, one repo
GITHUB_REPO = "your-user/fno-wave-inverse"   # repository to clone the pipeline from

MODE = "fit"         # "fit": auto-fit epochs to TRAIN_HOURS | "smoke": 3 epochs | "full": 300
TRAIN_HOURS = 5.0    # A100 hours stage C may spend training, when MODE = "fit"
FORCE_RETRAIN = False  # retrain even if checkpoints/full/history.json already exists
QUICK = True         # stage E/F statistics sizes: True = short pass, False = full (adds ~1-2 h)

assert MODE in ("fit", "smoke", "full"), MODE
print(f"MODE={MODE!r}  TRAIN_HOURS={TRAIN_HOURS}  FORCE_RETRAIN={FORCE_RETRAIN}  QUICK={QUICK}")
print(f"repo: {GITHUB_REPO}  token: "
      f"{'set' if GITHUB_TOKEN else 'MISSING -- clone will fail on a private repo'}")

## Getting the code and the data into this kernel

Three cases, in order: the repo is already at `/root/fno-wave-inverse` (the
headless launcher mounts it there, so this notebook runs headless unchanged);
it is found by walking up from the working directory (running locally from
inside a checkout); or neither, and it is cloned from GitHub -- with a tarball
fallback when the image has no `git`. The token is used for the clone and then
stripped from the stored remote.

The same idea for the data: wherever the notebook UI mounted the Volume, the
cell below finds it by the file it must hold (`datasets/train.h5`) and pins it
with `FNO_DATA_DIR` *before* `bootstrap.setup()` runs, so every stage agrees on
one data directory. `bootstrap.py` then puts the repo on `sys.path`, turns TF32
off, and reports the device -- it is the only platform-aware code in the
project, by design (see its docstring).

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO = pathlib.Path("/root/fno-wave-inverse")

if not (REPO / "src" / "config.py").exists():
    here = pathlib.Path.cwd()
    local = next((p for p in (here, *here.parents)
                  if (p / "src" / "config.py").exists()), None)
    if local is not None:
        REPO = local                       # local checkout, or headless launcher cwd

if (REPO / "src" / "config.py").exists():
    print(f"repo already present: {REPO}")
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    url = (f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git" if GITHUB_TOKEN
           else f"https://github.com/{GITHUB_REPO}.git")
    try:
        subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)],
                       check=True, capture_output=True)
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin",
                        f"https://github.com/{GITHUB_REPO}.git"],
                       check=True, capture_output=True)
        print(f"cloned {GITHUB_REPO} -> {REPO}  (token stripped from the remote)")
    except (FileNotFoundError, subprocess.CalledProcessError) as e:
        import tarfile
        import urllib.request
        print(f"git unavailable/failed ({e}) -- downloading the tarball instead")
        req = urllib.request.Request(
            f"https://api.github.com/repos/{GITHUB_REPO}/tarball/HEAD",
            headers={"Authorization": f"Bearer {GITHUB_TOKEN}"} if GITHUB_TOKEN else {})
        with urllib.request.urlopen(req) as r, tarfile.open(fileobj=r, mode="r|gz") as tar:
            root = tar.next().name.split("/")[0]
            tar.extractall(REPO.parent)
        (REPO.parent / root).rename(REPO)
        print(f"extracted {GITHUB_REPO} -> {REPO}")

os.environ["FNO_ROOT"] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

for cand in ("/vol/fno-data", "/mnt/fno-data", "/vol/fno-wave-inverse/data",
             "/mnt/fno-wave-inverse/data", "/volumes/fno-data",
             "/volumes/fno-wave-inverse-data", "/root/volumes/fno-data",
             "/root/fno-wave-inverse-data", "/root/data"):
    p = pathlib.Path(cand)
    if (p / "datasets" / "train.h5").exists():
        os.environ["FNO_DATA_DIR"] = str(p)
        print(f"data dir: {p}   (Volume found, pinned via FNO_DATA_DIR)")
        break
else:
    print("WARNING: no mounted directory holds datasets/train.h5.\n"
          "  Attach the Volume fno-wave-inverse-data at /vol/fno-data in the\n"
          "  notebook settings and re-run this cell. Stage B will refuse to run\n"
          "  without it; stage A (solver checks) needs no data and can proceed.")

In [ ]:
import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later stages read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

---
# Stage A -- Solver validation  (§11.2 steps 1-5)

The five sanity checks of §3.7 plus the two things that have to hold before any
of them mean anything: the configuration is internally consistent, and the
velocity-to-displacement deconvolution is conditioned across the whole operating
band.

§11.3 lists the solver sanity checks under *never cut, regardless of time*. The
reason is not diligence for its own sake. A surrogate trained against an
unvalidated solver is a surrogate for the wrong operator, and every metric
downstream -- relative L2, phase error, inversion success rate -- will look
fine while being an accurate report on the wrong physics. Nothing later in the
pipeline can detect that. This stage is the only place that can.

**Runtime.** Checks 1-3 are seconds. Checks 4 and 5 refine the grid and sweep a
radius, so they want a GPU: 20-40 minutes on an A100. `run_all` skips them on
CPU, loudly.

In [ ]:
cfg.self_check()

In [ ]:
from src.models.fno2d import band_in_modes

k_needed = band_in_modes()
print(f"parameter budget (primary variant)")
print(f"  config.total_params(d_v={cfg.D_V}, kmax={cfg.KMAX}, "
      f"n_blocks={cfg.N_BLOCKS}) = {cfg.total_params():,}")
print(f"\nmode truncation")
print(f"  band top f = {max(cfg.FREQS)/cfg.FC:.3f} f_c at nu = {min(cfg.NU_LIST)} needs "
      f"mode index {k_needed:.1f}")
print(f"  KMAX = {cfg.KMAX}  ->  {cfg.KMAX/k_needed:.2f}x headroom for near-field content")
print(f"  Nyquist on the {cfg.N_NET}^2 grid is {cfg.K_NYQUIST}, so the retained band is "
      f"strictly inside the resolved band")

## The deconvolution has to be conditioned, or steps 1-5 are measuring noise

The solver is a velocity-stress FDTD and runs a DFT inside the time loop, so what
comes out is a *velocity* phasor. Everything downstream -- the network's targets,
the incident cache, the inversion's data -- is a *displacement* phasor. The
bridge is

    u_hat(omega) = v_hat(omega) / (i omega s_hat(omega))

and dividing by `s_hat` is a deconvolution. The 5-cycle Hann-windowed tone burst
has spectral nulls at 0.6 f_c and 1.4 f_c; the band runs 0.66 to 1.34 f_c, which
sits *inside* those nulls but not far inside. `conditioning_report` measures how
much each line is amplified relative to the strongest one, and
`assert_conditioned` refuses anything worse than `MAX_DECONV_AMPLIFICATION = 25`.

This is one of the three named failure modes of the time-harmonic reduction (the
others being quadrature mismatch, handled by the midpoint rule at
t = (n + 1/2) dt, and wrap-around, which check 3 and stage B's tail-energy
diagnostic cover). It is checked first because a band-edge line amplified 200x
would turn every subsequent number in this notebook into a report on round-off.

In [ ]:
from src.solver import harmonic as H

rep = H.conditioning_report()
f = _np(rep["freqs"])
s = _np(rep["s_hat_abs"])
a = _np(rep["amplification"])
worst_i = int(rep["worst_index"])

print(f"{'m':>3} {'f/f_c':>8} {'|s_hat|':>12} {'amplification':>14}")
for m in range(len(f)):
    flag = "   <-- worst" if m == worst_i else ""
    print(f"{m:>3} {f[m]:8.4f} {s[m]:12.4e} {a[m]:14.3f}{flag}")

print(f"\nworst amplification {float(rep['worst']):.2f} at m = {worst_i} "
      f"(f = {f[worst_i]:.3f} f_c), limit {H.MAX_DECONV_AMPLIFICATION}")
H.assert_conditioned()
print("PASS  the deconvolution is conditioned across the whole band")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.1))

ff = np.linspace(0.3, 1.7, 601)
om = torch.tensor(2.0 * np.pi * ff, dtype=torch.float64)
sh = H.source_hat(om).abs().numpy()
ax[0].semilogy(ff, sh / sh.max(), lw=1.0, color="0.35",
               label="|s_hat| (continuous)")
ax[0].semilogy(f, s / sh.max(), "o", ms=4, color="C0", label="the 20 band lines")
for x in (0.6, 1.4):
    ax[0].axvline(x, ls=":", c="C3", lw=1.0)
ax[0].axvspan(min(f), max(f), color="C0", alpha=0.08)
ax[0].set(xlabel="f / f_c", ylabel="|s_hat| (normalised)", ylim=(1e-4, 2.0),
          title="tone-burst spectrum\n(Hann nulls at 0.6 and 1.4 f_c, dotted)")
ax[0].legend(fontsize=7.5, loc="lower center")

ax[1].plot(f, a, "o-", ms=4)
ax[1].axhline(H.MAX_DECONV_AMPLIFICATION, ls="--", c="C3", lw=1.0,
              label=f"limit = {H.MAX_DECONV_AMPLIFICATION:g}")
ax[1].set(xlabel="f / f_c", ylabel="1 / |s_hat| relative to the strongest line",
          title="deconvolution amplification per line")
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "01_deconvolution_conditioning.png")
plt.show()

## §11.2 steps 1-5

| # | check | gate | what a failure would mean |
|---|-------|------|---------------------------|
| 1 | energy drift with the absorber removed | `< 0.5%` over the run | the update is not conservative: wrong Lame coefficients, a stencil bug, or dt above the stability limit |
| 2 | P and S arrival times against analytic | within one time step | the wave speeds are wrong, or the source is not where the acquisition table says it is |
| 3 | residual energy after the wave has left | `< 1e-4` of peak | the PML is reflecting; every A-scan carries a ghost of itself |
| 4 | Rayleigh scaling of scattered energy with radius | slope in `(3.3, 4.7)` **and** visible mode conversion | small voids are not being resolved -- the label floor is above the smallest defect in the dataset |
| 5 | grid convergence, `256^2` vs `512^2` | `< 2%` rel-L2 | the training labels are discretisation error, not physics |

Check 4's gate is a conjunction, and deliberately so. The theoretical slope is 4
(energy goes as R^4 in the 2D Rayleigh limit, amplitude as R^2), the window
brackets it, and a run that lands the slope but produces no mode conversion has
almost certainly done so by scattering off a numerical artefact rather than off
the void.

In [ ]:
from src.solver import validate as V

include_slow = DEV.startswith("cuda")
if not include_slow:
    print("NO GPU DETECTED.\n"
          "Checks 4 and 5 refine the grid and will be skipped, so this run reports\n"
          "3 of 5.  Both are on the never-cut list of §11.3 -- do not quote any\n"
          "result from a CPU-only run of this stage.\n")

t0 = time.perf_counter()
results = V.run_all(device=DEV, include_slow=include_slow)
wall = time.perf_counter() - t0

print(f"\n{len(results)} checks in {wall/60:.1f} min on {DEV}\n")
for r in results:
    print(r)

by = {r.name.split(".")[0]: r for r in results}
n_pass = sum(1 for r in results if r.passed)
print(f"\n{n_pass} / {len(results)} passed")

In [ ]:
r = by.get("1")
if r is None:
    print("check 1 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    k0 = int(en.argmax())

    fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.0))
    ax[0].plot(t, en_n, lw=0.9)
    ax[0].axvline(t[k0], ls=":", c="0.5", lw=1.0)
    ax[0].set(xlabel="t / T_p", ylabel="E / E_max",
              title=f"total energy, L = {r.extras['l_domain']:g}, no absorber")

    tail = slice(k0, None)
    ax[1].plot(t[tail], en_n[tail] / en_n[tail][0] - 1.0, lw=0.9)
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.1e}")
    ax[1].axhline(-r.gate, ls="--", c="C3", lw=1.0)
    ax[1].set(xlabel="t / T_p", ylabel="E(t)/E(peak) - 1",
              title=f"drift {r.value:.2e}   oscillation "
                    f"{float(r.extras['oscillation']):.2e}")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check1_energy.png")
    plt.show()

In [ ]:
r = by.get("2")
if r is None:
    print("check 2 not in this run")
else:
    d = _np(r.extras["distances"])            # all 32 receivers
    ep = _np(r.extras["errs_p"])              # only the ones actually picked
    es = _np(r.extras["errs_s"])
    use = _np(r.extras["usable"]).astype(bool)
    asc = _np(r.extras["ascans"])

    d_use = d[use]
    n = min(len(ep), len(es), len(d_use))
    if n < len(d_use):
        print(f"{len(d_use) - n} usable receiver(s) had no clean window "
              f"(the record ends before the predicted arrival); not plotted")

    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.2))
    ax[0].plot(d_use[:n], ep[:n], "o", ms=4, label="P")
    ax[0].plot(d_use[:n], es[:n], "s", ms=4, label="S")
    if (~use).any():
        ax[0].axvspan(0.0, float(d[~use].max()), color="0.88", zorder=0,
                      label=f"excluded, P/S overlap ({int((~use).sum())} recv)")
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:g} step")
    ax[0].set(xlabel="source-receiver distance / lambda_p",
              ylabel="|arrival error| (time steps)",
              xlim=(0.0, 1.02 * float(d.max())), ylim=(0.0, None),
              title=f"worst |error| = {r.value:.3f} steps")
    ax[0].legend(fontsize=7.5)

    trace = asc[0] if asc.ndim == 4 else asc
    pick = np.argsort(d)[::-1][:4]
    t_ax = np.arange(trace.shape[-1]) * cfg.DT
    for k, i in enumerate(pick):
        x = trace[i, 0]
        env = _np(H.envelope(torch.from_numpy(np.ascontiguousarray(x))))
        off = 1.15 * k
        norm = max(abs(x).max(), 1e-30)
        ax[1].plot(t_ax, x / norm + off, lw=0.7, c=f"C{k}")
        ax[1].plot(t_ax, env / norm + off, lw=1.0, c="0.25", alpha=0.8)
    ax[1].set(xlabel="t / T_p", ylabel="receiver (offset)", yticks=[],
              title="x-velocity and analytic envelope\n(4 most distant receivers)")
    fig.tight_layout()
    savefig(fig, "01_check2_arrivals.png")
    plt.show()

In [ ]:
r = by.get("3")
if r is None:
    print("check 3 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    frac = float(r.extras["tail_fraction"])

    fig, ax = plt.subplots(figsize=(5.6, 3.1))
    ax.semilogy(t, np.maximum(en_n, 1e-16), lw=0.9)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0e}")
    ax.axvspan(t[int(0.9 * len(t))], t[-1], color="C3", alpha=0.08,
               label="tail window")
    ax.set(xlabel="t / T_p", ylabel="E(t) / E_peak",
           title=f"absorber residual {r.value:.2e}   tail fraction {frac:.2e}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check3_absorber.png")
    plt.show()

In [ ]:
r = by.get("4")
if r is None:
    print("check 4 not in this run (needs a GPU)")
else:
    en = _np(r.extras["energy"])
    kR = _np(r.extras["kR"])
    slope = float(r.extras["slope"])
    rad = kR / (2.0 * np.pi)
    loc_reported = float(r.extras["local_slope"])
    conv = float(r.extras["mode_conversion"])

    lp = np.diff(np.log(en)) / np.diff(np.log(rad))
    mid = np.sqrt(rad[1:] * rad[:-1])

    fig, ax = plt.subplots(1, 3, figsize=(11.0, 3.1))

    ax[0].loglog(rad, en, "o-", ms=4)
    ref = en[0] * (rad / rad[0]) ** 4.0
    ax[0].loglog(rad, ref, ls="--", c="0.45", lw=1.0, label="slope 4 (Rayleigh)")
    ax[0].set(xlabel="R / lambda_s", ylabel="scattered energy",
              title=f"fitted slope {slope:.3f}\ngate window (3.3, 4.7)")
    ax[0].legend(fontsize=8)

    ax[1].semilogx(mid, lp, "o-", ms=4, label="consecutive pairs")
    ax[1].axhspan(3.3, 4.7, color="C2", alpha=0.12, label="gate window")
    ax[1].axhline(4.0, ls=":", c="0.4", lw=1.0)
    ax[1].plot([mid[0]], [loc_reported], "x", c="C3", ms=9, mew=1.6,
               label=f"reported {loc_reported:.2f}")
    ax[1].set(xlabel="R / lambda_s (pair midpoint)", ylabel="d log E / d log R",
              title="local slope, pair by pair")
    ax[1].legend(fontsize=7.5)

    ax[2].bar([0], [conv], width=0.55, color="C0", alpha=0.85)
    ax[2].axhline(0.01, ls="--", c="C3", lw=1.0, label="gate: > 1%")
    ax[2].set(xticks=[0], xticklabels=[f"R = {rad[-1]:.3f} lambda_s"],
              ylabel="S-window energy / P-window energy",
              yscale="log", xlim=(-0.6, 0.6),
              title=f"mode conversion {conv:.1%}\nkR in "
                    f"[{kR.min():.2f}, {kR.max():.2f}]")
    ax[2].legend(fontsize=8)

    fig.tight_layout()
    savefig(fig, "01_check4_rayleigh.png")
    plt.show()
    print(f"grid {r.extras['grid']}  dx {float(r.extras['dx']):.5f}  "
          f"nt {int(r.extras['nt'])}")

In [ ]:
r = by.get("5")
if r is None:
    print("check 5 not in this run (needs a GPU)")
else:
    pf = _np(r.extras["per_frequency"]).reshape(-1)
    fr = np.asarray(cfg.FREQS[:len(pf)])

    fig, ax = plt.subplots(figsize=(6.4, 3.1))
    ax.bar(fr, pf, width=0.9 * cfg.DF, color="C0", alpha=0.85)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax.set(xlabel="f / f_c", ylabel="rel-L2, 256^2 vs 512^2",
           title=f"grid convergence at R = {float(r.extras['radius']):.3f}, "
                 f"refine {int(r.extras['refine'])}x\nworst line {pf.max():.4f}, "
                 f"reported {r.value:.4f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check5_convergence.png")
    plt.show()

In [ ]:
rows = [(r.name, f"{r.value:.4e}", f"{r.gate:.2e}", r.units or "-",
         "PASS" if r.passed else "FAIL") for r in results]
table(rows, ["check", "value", "gate", "units", ""])

record = {
    "device": DEV,
    "gpu": E.gpu_name,
    "include_slow": include_slow,
    "wall_minutes": wall / 60.0,
    "deconvolution": {"worst_amplification": float(rep["worst"]),
                      "worst_index": worst_i,
                      "limit": H.MAX_DECONV_AMPLIFICATION},
    "checks": {r.name: dict(passed=bool(r.passed), value=float(r.value),
                            gate=float(r.gate), units=r.units, detail=r.detail)
               for r in results},
    "n_pass": n_pass,
    "n_total": len(results),
}
dump(record, "01_solver_validation.json")

if not include_slow:
    print("\nINCOMPLETE: checks 4 and 5 were skipped.  Re-run on a GPU.")
elif n_pass == len(results):
    print("\nAll five solver checks pass.  The dataset on the Volume is worth training on.")
else:
    print("\nSTOP.  Fix the solver before training -- a surrogate trained on these\n"
          "labels would be an accurate model of the wrong operator.")

### If a gate fails

- **1 (energy).** Check `dt` against `CFL_LIMIT_4TH * CFL_SAFETY * dx / c_p` first; a
  marginally unstable run drifts slowly rather than exploding. Then check the Lame
  coefficients: `lame_from_nu` assumes `rho = c_p = 1`.
- **2 (arrivals).** Almost always the source position. `source_position` documents a
  deliberate `dx_fine/2` offset between the network-grid index and the physical
  coordinate; using the wrong one shifts every arrival by half a fine cell.
- **3 (absorber).** Increase `N_PML_FINE`, or reduce `pml_d0`. A residual just above the
  gate is usually the profile's grading exponent, not its thickness.
- **4 (Rayleigh).** If the slope is right but conversion is absent, the interface width
  `EPS_INTERFACE_CELLS` is too large relative to the radius being tested; if both fail at
  small R only, that is the label floor and `R_MIN_LS` needs raising.
- **5 (convergence).** Failing only at the top of the band means the network grid is
  under-resolved there; that is an argument for trimming `M_FREQ`.

---
# Stage B -- Verifying the uploaded dataset  (§11.2 step 6, post-hoc)

The dataset was generated on another GPU and uploaded to the Volume. This stage
runs everything notebook 02 runs *after* generation: the size projection, the
re-solve spot check, the wrap-around diagnostic, and the loader end-to-end. The
expensive part -- 2800 forward solves -- has already been paid for elsewhere;
what is left is the part that catches a dataset that is subtly wrong, because
such a dataset still trains, still converges, and produces a surrogate for an
operator nobody asked for.

One caveat that comes with cross-hardware verification: the spot check re-solves
stored samples on *this* GPU and compares against phasors produced on *another*
one. The gate is `rel-L2 < 1e-5` against round-off, which same-hardware runs
meet with a wide margin; different GPUs may differ in fused-multiply-add usage
by small amounts per step, accumulated over 1408 steps. Agreement at the
`1e-6`-ish level confirms integrity; anything at `1e-3` and up is not
round-off, it is a genuinely different run, and the file should not be trained
on.

In [ ]:
import h5py

from src.data import generate as G

assert all(paths[k].exists() for k in ("train", "val", "test")), \
    "the splits are not all on the Volume -- see the upload commands at the top"

# assert_compatible refuses to read a dataset generated under a different config.
# It compares every key in generate._SNAPSHOT_KEYS, so a change to the grid, the
# band, the time step or the interface width invalidates the file rather than
# silently shifting what the labels mean.  This is the check that catches
# "generated on a machine with a different checkout".
N, rows = {}, []
for split, p in paths.items():
    with h5py.File(p, "r") as f:
        G.assert_compatible(f)
        th = f["samples/theta"][:]
        si = f["samples/src_idx"][:]
        N[split] = int(f.attrs["n_samples"])
        rows.append((split, f"{N[split]}",
                     f"{p.stat().st_size/1e9:.2f} GB",
                     f"{th[:, 2].min():.3f}-{th[:, 2].max():.3f}",
                     f"{sorted(set(int(v) for v in si))}",
                     f"{float(f.attrs['generation_seconds'])/60:.0f} min"))
table(rows, ["split", "n", "size", "R range", "sources used", "gen wall"])

with h5py.File(paths["train"], "r") as f:
    assert list(f.attrs["src_pool"]) == list(cfg.SRC_TRAIN), \
        "train src_pool is not SRC_TRAIN"
with h5py.File(paths["test"], "r") as f:
    assert list(f.attrs["src_pool"]) == list(range(cfg.N_SRC)), \
        "test src_pool must be all 8 sources"
print(f"\nsource pools verified: train/val exclude SRC_HELDOUT = {cfg.SRC_HELDOUT}, "
      f"test draws from all {cfg.N_SRC}")

N_TOTAL = sum(N.values())
print(f"\nsplits: {N}   total {N_TOTAL} samples\n")
G.print_projection(N_TOTAL)

with h5py.File(paths["train"], "r") as f:
    print("\nfile layout")
    f.visit(lambda k: print("  ", k))

In [ ]:
import shutil

# The projection's clock is *this* GPU, measured rather than asserted -- the same
# number stage D later divides the surrogate's forward-call time by, and the
# honest cross-check of the generation machine's own clock.
thr = G.calibrate(device=DEV)
print(f"measured throughput {thr/1e9:.2f} G cell-steps/s on {E.gpu_name}\n")
G.print_projection(N_TOTAL, throughput_cell_steps_per_s=thr)

need = G.projected_size(N_TOTAL)["total"]
du = shutil.disk_usage(E.data)
print(f"\ndisk at {E.data}")
print(f"  free  {du.free/1e9:8.2f} GB   (dataset itself is {need/1e9:.2f} GB, already on the Volume)")
print(f"  {'OK' if du.free > 0.25 * need else 'TIGHT -- checkpoints need ~120 MB'}")

## The incident field, and its two normalisation scales

The incident field depends only on (source index, nu), never on the defect, so
there are 8 x 4 = 32 solves cached in every split file, shared by every sample
that uses that acquisition. Caching them makes the scattered field `u_tot -
u_inc` an exact difference of two solves on the same grid with the same time
stepping.

Two normalisation scales come out of this, and the distinction matters:

- `scale` is `max|u_inc|` over the whole domain, dominated by the near-source
  singularity. This is what the network's inputs and targets are divided by.
- `scale_recv` is `max|u_inc|` over the 32-receiver ring, three cells inside the
  absorber, where the field has already spread. This is what the receiver misfit
  uses.

They differ by about two orders of magnitude. Using the domain scale for the
receiver misfit would divide the residual by a number 100x too large and make
the measurement term silently negligible -- the loss would still go down, and
the ring would be unconstrained.

In [ ]:
with h5py.File(paths["test"], "r") as f:
    inc = {k: f[f"incident/{k}"][:] for k in ("phasors", "ascans", "scale", "scale_recv")}
for k, v in inc.items():
    print(f"  {k:>11}  {str(v.shape):>28}  {v.dtype}")

In [ ]:
sc, scr = inc["scale"], inc["scale_recv"]
fr = np.asarray(cfg.FREQS)

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
for s in range(cfg.N_SRC):
    ax[0].semilogy(fr, sc[s, 0], lw=0.8, alpha=0.8)
    ax[0].semilogy(fr, scr[s, 0], lw=0.8, alpha=0.8, ls="--")
ax[0].semilogy([], [], c="0.3", lw=0.8, label="domain max (solid)")
ax[0].semilogy([], [], c="0.3", lw=0.8, ls="--", label="receiver-ring max (dashed)")
ax[0].set(xlabel="f / f_c", ylabel="max |u_inc|",
          title=f"incident amplitude, nu = {cfg.NU_LIST[0]}, all 8 sources")
ax[0].legend(fontsize=7.5)

ratio = sc / np.maximum(scr, 1e-30)
ax[1].semilogy(fr, ratio.reshape(-1, len(fr)).T, lw=0.7, alpha=0.6)
ax[1].set(xlabel="f / f_c", ylabel="domain max / ring max",
          title=f"ratio of the two scales\nmedian {np.median(ratio):.0f}x over all "
                f"{ratio.shape[0]*ratio.shape[1]} (source, nu)")
fig.tight_layout()
savefig(fig, "02_incident_scales.png")
plt.show()

print(f"ratio: min {ratio.min():.1f}  median {np.median(ratio):.1f}  "
      f"max {ratio.max():.1f}")

## §11.2 step 6: re-run the solver and compare

The check that catches everything the layout inspection above cannot. Twenty
stored samples are re-solved from their stored `theta`, `src_idx` and `nu_idx`
alone, the *stored* incident phasors are subtracted, and the result is compared
to the stored scattered phasors. The gate is `rel-L2 < 1e-5`, which is far
tighter than any physics tolerance and is deliberately so: the solver is
deterministic, so anything above round-off means the file does not describe the
run that produced it.

If this fails *badly* (orders of magnitude), the likely causes in order: the
incident cache indexed with `(nu_idx, src_idx)` instead of `(src_idx, nu_idx)`;
`nu_idx` read as a Poisson ratio rather than an index into `NU_LIST`; or the
component and frequency axes transposed on write. If it fails *marginally* on
cross-hardware round-off, see the caveat at the top of this stage.

In [ ]:
from src.solver.fdtd_elastic import ElasticFDTD2D
from src.solver import harmonic as H

N_SPOT = 20
SPOT_BATCH = 4

with h5py.File(paths["test"] if paths["test"].exists() else paths["train"], "r") as f:
    n = int(f.attrs["n_samples"])
    pick = np.linspace(0, n - 1, N_SPOT).astype(int)
    th_all = f["samples/theta"][:][pick]
    s_all = f["samples/src_idx"][:][pick].astype(int)
    j_all = f["samples/nu_idx"][:][pick].astype(int)
    inc_ph = f["incident/phasors"][:]
    stored = np.stack([f["samples/us_phasors"][int(i)] for i in pick])

om = H.omegas_tensor(DEV)
errs = []
for lo in tqdm(range(0, N_SPOT, SPOT_BATCH), desc="re-solving"):
    hi = min(N_SPOT, lo + SPOT_BATCH)
    th = torch.as_tensor(th_all[lo:hi], device=DEV)
    nu_vals = torch.tensor([cfg.NU_LIST[j] for j in j_all[lo:hi]], device=DEV)
    chi = G.fine_chi(th, device=DEV)
    lam, mu, rho = G._materials(chi, nu_vals)
    sim = ElasticFDTD2D(lam, mu, rho)
    src = [cfg.net_to_fine(*cfg.SOURCES_NET[s]) for s in s_all[lo:hi]]
    res = sim.run(src, nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
    u_tot = H.displacement_from_field(res.phasors, omegas=om)
    u_inc = torch.as_tensor(
        np.stack([inc_ph[s, j] for s, j in zip(s_all[lo:hi], j_all[lo:hi])]),
        device=DEV)
    fresh = (u_tot - u_inc).cpu().numpy()
    ref = stored[lo:hi]
    num = np.linalg.norm((fresh - ref).reshape(hi - lo, -1), axis=1)
    den = np.linalg.norm(ref.reshape(hi - lo, -1), axis=1)
    errs.extend((num / np.maximum(den, 1e-30)).tolist())

errs = np.asarray(errs)
GATE_SPOT = 1e-5
print(f"\nrel-L2 over {N_SPOT} re-solved samples")
print(f"  median {np.median(errs):.3e}   max {errs.max():.3e}   gate {GATE_SPOT:.0e}")
print(f"  {'PASS' if errs.max() < GATE_SPOT else 'FAIL'}  "
      f"the file describes the run that produced it")

In [ ]:
i_worst = int(errs.argmax())
u_ref = stored[i_worst]                              # [2, M, ny, nx] complex
m_show = [0, cfg.M_FREQ // 2, cfg.M_FREQ - 1]

fig, ax = plt.subplots(2, 4, figsize=(11.5, 5.4))
for c, m in enumerate(m_show):
    a = np.abs(u_ref[0, m])
    ax[0, c].imshow(a, origin="lower", cmap="magma")
    ax[0, c].set_title(f"|u_s,x|  f = {cfg.FREQS[m]:.2f} f_c", fontsize=8)
    ax[1, c].imshow(np.angle(u_ref[0, m]), origin="lower", cmap="twilight",
                    vmin=-np.pi, vmax=np.pi)
    ax[1, c].set_title("arg u_s,x", fontsize=8)
for a_ in ax.ravel():
    a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)

ax[0, 3].semilogy(range(len(errs)), np.sort(errs)[::-1], "o-", ms=3)
ax[0, 3].axhline(GATE_SPOT, ls="--", c="C3", lw=1.0, label="gate 1e-5")
ax[0, 3].set(xlabel="sample (sorted)", ylabel="rel-L2 vs stored",
             title="re-solve agreement")
ax[0, 3].legend(fontsize=7.5)
ax[0, 3].grid(True, alpha=0.25)

amp = np.linalg.norm(stored.reshape(N_SPOT, -1), axis=1)
ax[1, 3].semilogy(th_all[:, 2], amp, "o", ms=4)
ax[1, 3].set(xlabel="R (physical)", ylabel="||u_s||_2",
             title="scattered amplitude carries R\n(this is why the target is not "
                   "self-normalised)")
ax[1, 3].grid(True, alpha=0.25)

fig.suptitle(f"stored sample {int(np.linspace(0, 1, N_SPOT)[i_worst]*100):d}th "
             f"percentile of the re-solve error", fontsize=9)
fig.tight_layout()
savefig(fig, "02_spot_check.png")
plt.show()

## The scattered field really is a difference, not a difference plus a constant

An independent check on the incident subtraction, and one that does not depend on
any stored array. Shrink the void radius towards zero at a fixed centre and watch
the scattered amplitude. In the Rayleigh regime the scattered amplitude goes as
`R^2`, so a log-log fit should give a slope near 2, and -- the point of the
test -- it should keep falling. A constant offset in the subtraction (wrong
source, wrong nu, an off-by-one in the cache index) is invisible at
`R = 0.8 lambda_s` where the scattered field is large, and shows up as a
**floor** here.

The smallest radii in the sweep are below the interface width
`EPS_INTERFACE_CELLS = 1.5` cells, so `chi` never reaches 1 and the "void" is a
soft dimple. That is not a defect of the test: the field still has to go to zero
smoothly, and it is the floor that is being looked for, not the exponent at the
last point.

This check needs the incident cache in numpy form; `inc` (loaded from the test
split above) is exactly that.

In [ ]:
radii_ls = np.array([0.40, 0.30, 0.20, 0.14, 0.10, 0.05, 0.025])
S_IDX, J_IDX = 0, 0
lam_s = cfg.cs_over_cp(cfg.NU_LIST[J_IDX]) / cfg.FC
xc, yc = 0.55 * cfg.L_DOMAIN, 0.45 * cfg.L_DOMAIN

th = torch.tensor([[xc, yc, float(r) * lam_s] for r in radii_ls],
                  dtype=torch.float32, device=DEV)
nu_vals = torch.full((len(radii_ls),), cfg.NU_LIST[J_IDX], device=DEV)
chi = G.fine_chi(th, device=DEV)
lam, mu, rho = G._materials(chi, nu_vals)
sim = ElasticFDTD2D(lam, mu, rho)
res = sim.run([cfg.net_to_fine(*cfg.SOURCES_NET[S_IDX])] * len(radii_ls),
              nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
u_tot = H.displacement_from_field(res.phasors, omegas=om)
u_inc1 = torch.as_tensor(inc["phasors"][S_IDX, J_IDX], device=DEV).unsqueeze(0)
amp = (u_tot - u_inc1).abs().pow(2).sum(dim=(1, 2, 3, 4)).sqrt().cpu().numpy()

k = np.polyfit(np.log(radii_ls), np.log(amp), 1)
slope = float(k[0])
print(f"log-log slope of ||u_s|| vs R : {slope:.3f}   (Rayleigh amplitude ~ R^2)")
print(f"amplitude falls by {amp[0]/amp[-1]:.1f}x over a {radii_ls[0]/radii_ls[-1]:.0f}x "
      f"radius range -- no floor")

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.loglog(radii_ls, amp, "o-", ms=5, label="||u_s||_2")
ax.loglog(radii_ls, amp[0] * (radii_ls / radii_ls[0]) ** 2.0, ls="--", c="0.45",
          lw=1.0, label="slope 2")
ax.axvline(cfg.EPS_INTERFACE_CELLS * cfg.DX_NET / lam_s, ls=":", c="C3", lw=1.0,
           label="interface width")
ax.axvline(cfg.R_MIN_LS, ls="-.", c="C2", lw=1.0, label=f"R_MIN_LS = {cfg.R_MIN_LS}")
ax.set(xlabel="R / lambda_s", ylabel="scattered amplitude",
       title=f"incident subtraction has no offset\nfitted slope {slope:.2f}")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "02_incident_subtraction.png")
plt.show()

## Wrap-around: is the DFT window long enough?

The phasors come from a DFT run over a finite window of `NT = 1408` steps;
anything still ringing at the end of that window aliases back onto the band.
`tail_energy_fraction` measures the energy in the last 10% of each A-scan as a
fraction of the total. Check 3 of stage A established that the absorber works
on a clean domain; this measures the same thing on the real dataset, where a
void near the ring can hold energy longer than a homogeneous domain does.

In [ ]:
with h5py.File(paths["train"], "r") as f:
    a = torch.from_numpy(f["samples/ascans"][:64])
    th64 = f["samples/theta"][:64]
frac = _np(H.tail_energy_fraction(a))
frac = frac.reshape(frac.shape[0], -1).max(axis=1) if frac.ndim > 1 else frac

print(f"tail energy fraction over {len(frac)} samples")
print(f"  median {np.median(frac):.3e}   p90 {np.quantile(frac, 0.9):.3e}   "
      f"max {frac.max():.3e}")
print(f"  {'OK' if frac.max() < 1e-3 else 'HIGH -- lengthen T_END or thicken the PML'}")

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.0))
ax[0].hist(np.log10(np.maximum(frac, 1e-12)), bins=24, color="C0", alpha=0.85)
ax[0].set(xlabel="log10 tail energy fraction", ylabel="samples",
          title="wrap-around margin, 64 training samples")
tr = _np(a[int(frac.argmax())])
t_ax = np.arange(tr.shape[-1]) * cfg.DT
for r in range(0, tr.shape[0], 8):
    ax[1].plot(t_ax, tr[r, 0] / max(abs(tr[r, 0]).max(), 1e-30) + 1.1 * (r // 8),
               lw=0.6)
ax[1].axvspan(0.9 * t_ax[-1], t_ax[-1], color="C3", alpha=0.10, label="tail window")
ax[1].set(xlabel="t / T_p", yticks=[], ylabel="receiver (every 8th)",
          title=f"worst sample, R = {th64[int(frac.argmax()), 2]:.3f}")
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "02_wraparound.png")
plt.show()

## The loader, end to end

The path the training loop will actually take. Three things are being confirmed,
all of which have silent failure modes.

1. **Shapes and dtypes** through `WaveDataset` -> `make_loader` -> `batch_to_model`.
   The input has 12 channels, the target 4, and the frequency axis is folded
   into the batch, so `B = batch_size * n_freq`.
2. **Frequency subsetting is resampled per epoch.** `set_epoch` reseeds it, and
   that only works because `make_loader` sets `persistent_workers=False`. The
   symptom of it breaking would be a validation curve that plateaus early for
   no visible reason.
3. **Input and target share one divisor.** Both are divided by
   `incident/scale`, never by their own norm. Dividing the target by its own
   norm is the one normalisation that must not happen: the scattered amplitude
   carries the radius, and normalising it away leaves the inversion with
   nothing to recover `R` from.

In [ ]:
from src.data.dataset import WaveDataset, batch_to_model, make_loader

ds = WaveDataset(str(paths["train"]), train=True)
print(f"{len(ds)} samples, {ds.n_freq} of {cfg.M_FREQ} frequencies per sample per epoch")

ds.set_epoch(0)
s0 = ds[0]
ds.set_epoch(1)
s1 = ds[0]
print(f"\nsample 0 frequencies, epoch 0: {[round(float(v), 3) for v in s0['freqs']]}")
print(f"sample 0 frequencies, epoch 1: {[round(float(v), 3) for v in s1['freqs']]}")
same = torch.equal(s0["freqs"], s1["freqs"])
print(f"  {'FAIL -- set_epoch is not reseeding' if same else 'OK'}  subsets differ")

print("\nper-sample tensors")
for k, v in s0.items():
    print(f"  {k:>10}  {str(tuple(v.shape)):>22}  {str(v.dtype):>16}")

In [ ]:
ld = make_loader(ds, batch_size=4, num_workers=0)
batch = next(iter(ld))
x, y = batch_to_model(batch)
print(f"x {tuple(x.shape)} {x.dtype}      ({cfg.C_IN} channels expected)")
print(f"y {tuple(y.shape)} {y.dtype}      ({cfg.C_OUT} channels expected)")
print(f"rows = batch {4} x n_freq {ds.n_freq} = {4 * ds.n_freq}")
assert x.shape[1] == cfg.C_IN and y.shape[1] == cfg.C_OUT
assert x.shape[0] == y.shape[0] == 4 * ds.n_freq

from src import features as feat

names = feat.INPUT_CHANNELS
print("\nper-channel statistics of one batch")
table([(names[c], f"{float(x[:, c].mean()):+.4f}", f"{float(x[:, c].std()):.4f}",
        f"{float(x[:, c].min()):+.3f}", f"{float(x[:, c].max()):+.3f}")
       for c in range(x.shape[1])],
      ["channel", "mean", "std", "min", "max"])

rows_norm = y.flatten(1).norm(dim=1)
print(f"\ntarget row norms: min {float(rows_norm.min()):.4f}  "
      f"max {float(rows_norm.max()):.4f}  ratio {float(rows_norm.max()/rows_norm.min()):.1f}x")
print("A self-normalised target would make every one of these 1.0 and delete the "
      "radius.")
ds.close()

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "data_dir": str(E.data),
    "generated_elsewhere": True,
    "splits": {k: dict(n=N[k], path=str(paths[k]), exists=paths[k].exists(),
                       bytes=(paths[k].stat().st_size if paths[k].exists() else 0))
               for k in N},
    "projected_bytes": G.projected_size(N_TOTAL),
    "throughput_cell_steps_per_s": thr,
    "spot_check": dict(n=N_SPOT, median=float(np.median(errs)),
                       max=float(errs.max()), gate=GATE_SPOT,
                       passed=bool(errs.max() < GATE_SPOT)),
    "incident_subtraction_slope": slope,
    "tail_energy_fraction": dict(median=float(np.median(frac)), max=float(frac.max())),
    "scale_ratio_domain_over_ring": float(np.median(ratio)),
}
dump(record, "02_dataset_generation.json")
print("\nDataset verified.  Stage C trains on it.")

---
# Stage C -- Training the FNO surrogate  (§11.2 steps 7-8, one arm)

Before the loop starts, three things are established on a single batch, because
each has a failure mode that a converging training curve would hide:

- the parameter count is what §4.4 claims, and the radial mask is actually
  masking;
- each of the four loss terms is the size it is supposed to be relative to the
  others, and the physics term has a **measurable floor** -- the residual of the
  *true* field on the network grid, which no prediction can beat;
- `balance_alpha` returns something sane when measured on the projection head.

**This stage trains the `full` arm only** (α = balanced physics weight). The
`nophys` ablation arm doubles the cost and does not fit the session; run it
headless later and this notebook's thesis table picks it up:

    modal run modal_app.py::train_arm --arm nophys

Both arms write `checkpoints/<arm>/best.pt` + `history.json` on the Volume, and
this stage skips itself if `history.json` is already there (set
`FORCE_RETRAIN = True` in the first cell to override).

In [ ]:
EPOCHS = {"smoke": 3, "full": cfg.EPOCHS}.get(MODE)
WORKERS = 2           # make_loader's docstring: drop to 2 on smaller CPU allocations

assert paths["train"].exists() and paths["val"].exists(), \
    "stage B failed -- there is no dataset to train on"

hist_path = CKPT_DIR / "history.json"
have_hist = hist_path.exists() and not FORCE_RETRAIN
if have_hist:
    import json as _json
    _h = _json.loads(hist_path.read_text())
    print(f"checkpoints/full/history.json exists "
          f"({_h.get('final', {}).get('rel_l2', '??')} final rel_l2, "
          f"{len(_h.get('train', []))} epochs) -- skipping training")
print(f"\nMODE={MODE!r} -> EPOCHS={EPOCHS}  ({'measured probe below' if EPOCHS is None else ''})"
      f"  workers={WORKERS}")

## The operator

Four Fourier blocks at `d_v = 32`, `KMAX = 28` retained modes, radial
truncation, no normalisation layers, and the final block linear.

`effective_params` and `allocated_params` differ, and the difference is the
point of radial truncation. The weight tensors are `kmax x kmax` boxes, but the
mask keeps only `|k| <= kmax` on the integer lattice, so about `pi/4` of the box
survives -- a 21% saving with no loss of resolved bandwidth, because the corner
modes of the box are at `|k| = sqrt(2) kmax`, beyond anything the physics puts
energy in. The masked coefficients are zeroed at construction and re-masked on
every forward pass, so `effective_params` is the number of real parameters that
can actually move, and it is the number quoted.

In [ ]:
from src.models import fno2d

model = fno2d.build("primary").to(DEV)
print(model.summary())
print(f"\nconfig.total_params (independent arithmetic): {cfg.total_params():,}")
print(f"model.effective_params:                        {model.effective_params():,}")
print(f"model.allocated_params:                        {model.allocated_params():,}")
print(f"radial truncation saves "
      f"{1 - model.effective_params()/model.allocated_params():.1%} of the spectral "
      f"weights")
print(f"\nfinal block linear: act={model.blocks[-1].act} "
      f"(the others: {[b.act for b in model.blocks[:-1]]})")

In [ ]:
sc = model.blocks[0].spectral
m1 = _np(sc.m1)[0, 0]
m2 = _np(sc.m2)[0, 0]

fig, ax = plt.subplots(1, 3, figsize=(10.2, 3.0))
ax[0].imshow(m1, origin="lower", cmap="Greys_r", interpolation="nearest")
ax[0].set(title=f"mask, k_y >= 0 block\n{int(m1.sum())} of {m1.size} modes kept",
          xlabel="k_x", ylabel="k_y")
ax[1].imshow(m2, origin="lower", cmap="Greys_r", interpolation="nearest")
ax[1].set(title="mask, k_y < 0 block\n(row iy is k_y = -(kmax - iy))", xlabel="k_x")

kk = np.hypot(*np.meshgrid(np.arange(sc.kmax), np.arange(sc.kmax), indexing="ij"))
w = np.abs(_np(sc.w1)[0, 0])
ax[2].semilogy(kk[m1 > 0].ravel(), np.maximum(w[m1 > 0].ravel(), 1e-12), ".",
               ms=2, alpha=0.5, label="kept")
if (m1 == 0).any():
    ax[2].semilogy(kk[m1 == 0].ravel(), np.maximum(w[m1 == 0].ravel(), 1e-12), ".",
                   ms=2, c="C3", alpha=0.6, label="masked (exactly 0)")
ax[2].axvline(sc.kmax, ls="--", c="0.4", lw=1.0, label=f"|k| = KMAX = {sc.kmax}")
ax[2].set(xlabel="|k|", ylabel="|w| at init", title="masked weights are zero, not small")
ax[2].legend(fontsize=7.5)
for a_ in ax[:2]:
    a_.grid(False)
fig.tight_layout()
savefig(fig, "03_radial_mask.png")
plt.show()

## The four loss terms on one batch

    L = L_field + gamma L_H1 + beta L_meas + alpha L_phys

The table below evaluates the same batch four ways, adding one term at a time.
Two rows matter more than the rest:

- **prediction = 0** (the untrained network outputs almost nothing). `L_field`
  is then 1.0 by construction, and `L_phys` is the residual of `u_inc` alone in
  the presence of the void -- which is exactly the scattering source term the
  network is being asked to cancel. A near-zero value here would mean the
  physics term cannot see the defect at all.
- **prediction = target** (the labels themselves). `L_field`, `L_H1` and
  `L_meas` are exactly 0; `L_phys` is *not*, and its value is the floor. The
  residual is evaluated on the `128^2` network grid at half the solver's
  resolution, so the 4th-order stencil's own dispersion error and the
  1.5-cell coefficient transition at the void boundary set a floor the network
  cannot go below no matter how right it is. Every `L_phys` printed during
  training should be read against this number, not against zero.

That floor is also the argument for balancing `alpha` instead of fixing it.

In [ ]:
from src import features as feat
from src import losses as L
from src import training
from src.data.dataset import WaveDataset, batch_to_model, make_loader, to_device

ds = WaveDataset(str(paths["train"]), train=True)
ds.set_epoch(0)
batch = to_device(next(iter(make_loader(ds, batch_size=8, num_workers=0))), DEV)
x, y = batch_to_model(batch)
recv = training.receivers_tensor(DEV)
ctx = L.make_context(batch["chi"], batch["nu"], batch["freqs"], batch["src_idx"])
u_inc = feat.flatten_freq(batch["u_inc"])
print(f"batch: x {tuple(x.shape)}  y {tuple(y.shape)}  "
      f"ctx.weight {tuple(ctx.weight.shape)}")

In [ ]:
def terms_of(pred, **kw):
    return L.compute(pred, y, recv_yx=recv, ctx=ctx, u_inc=u_inc,
                     generator=torch.Generator().manual_seed(cfg.SEED), **kw)

with torch.no_grad():
    pred0 = model(x)
    combos = [
        ("field only",          dict(gamma=0.0, beta=0.0, alpha=0.0)),
        ("+ H1",                dict(gamma=cfg.GAMMA_H1, beta=0.0, alpha=0.0)),
        ("+ ring",              dict(gamma=cfg.GAMMA_H1, beta=cfg.BETA_MEAS, alpha=0.0)),
        ("+ physics (default)", dict(gamma=cfg.GAMMA_H1, beta=cfg.BETA_MEAS,
                                     alpha=cfg.ALPHA_PHYS)),
    ]
    rows = []
    for label, kw in combos:
        t = terms_of(pred0, **kw)
        rows.append((label, f"{float(t.total):.4f}", f"{float(t.field):.4f}",
                     f"{float(t.h1):.4f}", f"{float(t.meas):.4f}",
                     f"{float(t.phys):.4f}", f"{t.alpha:.1e}"))
    t_zero = terms_of(torch.zeros_like(pred0), **combos[-1][1])
    t_true = terms_of(y.clone(), **combos[-1][1])

table(rows, ["untrained model", "total", "field", "H1", "meas", "phys", "alpha"])
print()
table([("prediction = 0", f"{float(t_zero.field):.4f}", f"{float(t_zero.h1):.4f}",
        f"{float(t_zero.meas):.4f}", f"{float(t_zero.phys):.6f}"),
       ("prediction = target", f"{float(t_true.field):.4f}", f"{float(t_true.h1):.4f}",
        f"{float(t_true.meas):.4f}", f"{float(t_true.phys):.6f}")],
      ["reference point", "field", "H1", "meas", "phys"])

PHYS_FLOOR = float(t_true.phys)
print(f"\nL_phys floor (true field on the 128^2 grid): {PHYS_FLOOR:.6f}")
print(f"L_phys with no scattered field at all:      {float(t_zero.phys):.6f}")
print(f"ratio {float(t_zero.phys)/max(PHYS_FLOOR, 1e-30):.1f}x -- the physics term can "
      f"see the defect")

## `balance_alpha` on the projection head

`alpha` such that `alpha ||grad L_phys|| = 0.1 ||grad L_data||`, measured on
`model.project.parameters()` -- the two 1x1 convolutions of the projection head,
and the last place the two gradients are still the same kind of quantity.

It costs two extra backward passes on a retained graph, which is why the
training loop calls it every 200 steps and not every step, and calls it *before*
`opt.step()`: the retained graph's saved activations belong to the current
parameters, so balancing after the step would measure a ratio at a point the
network is no longer at.

In [ ]:
pred = model(x)
t = L.compute(pred, y, recv_yx=recv, ctx=ctx, u_inc=u_inc, alpha=cfg.ALPHA_PHYS,
              generator=torch.Generator().manual_seed(cfg.SEED))
l_data = t.field + cfg.GAMMA_H1 * t.h1 + cfg.BETA_MEAS * t.meas
balance_params = [p for p in model.project.parameters() if p.requires_grad]
print(f"balancing on {len(balance_params)} tensors, "
      f"{sum(p.numel() for p in balance_params):,} parameters "
      f"(the projection head)")

a_fresh = L.balance_alpha(l_data, t.phys, balance_params)
a_ema = L.balance_alpha(l_data, t.phys, balance_params, alpha_prev=cfg.ALPHA_PHYS)
print(f"\nalpha from the gradient-norm ratio : {a_fresh:.4e}")
print(f"alpha EMA-smoothed from {cfg.ALPHA_PHYS:.0e}     : {a_ema:.4e}")
print(f"config default ALPHA_PHYS         : {cfg.ALPHA_PHYS:.4e}")
print(f"clamp                             : (1e-5, 1.0)")

a_all = L.balance_alpha(l_data, t.phys,
                        [p for p in model.parameters() if p.requires_grad],
                        alpha_prev=None)
print(f"\nsame ratio measured on ALL parameters: {a_all:.4e} "
      f"({a_all/max(a_fresh, 1e-30):.2f}x different)")
ds.close()
del pred, t, l_data

## How many epochs does the budget buy?

`EPOCHS = 300` on 2000 samples is 8-16 h on an A100 -- the full schedule of
§7.5, and too much for a session that also owes time to stages D-F. Rather than
guessing, the cell below *measures* this GPU's per-epoch wall clock with a
2-epoch probe run, then fits

    epochs = floor(TRAIN_HOURS * 3600 / measured_seconds_per_epoch)

The probe writes into `checkpoints/probe/` and is deleted afterwards; the real
run then starts from a clean seed, with the cosine schedule defined over the
fitted epoch count (the schedule is part of the method -- it cannot be
restarted mid-run, which is also why one arm per session is the honest unit).

Two things to read from the fitted number:

- if it comes out **below ~150**, the forward gates in stage D may land
  marginally above 5% rel-L2 -- that is the budget, not the architecture;
  re-run stages D-F after a headless 300-epoch run;
- if it comes out **at or above 300**, the budget covers the full schedule and
  the cap is `cfg.EPOCHS` anyway.

In [ ]:
if have_hist:
    hist = _h
    EPOCHS_DONE = len(hist.get("train", []))
    print(f"training skipped: {EPOCHS_DONE} epochs already on the Volume")
elif EPOCHS is None:
    import shutil as _sh

    probe_dir = E.checkpoints / "probe"
    t0 = time.perf_counter()
    _m = fno2d.build("primary")
    training.train(_m, str(paths["train"]), str(paths["val"]),
                   out_dir=str(probe_dir), device=DEV, epochs=2,
                   num_workers=WORKERS, alpha=cfg.ALPHA_PHYS)
    secs_per_ep = (time.perf_counter() - t0) / 2
    _sh.rmtree(probe_dir)

    EPOCHS = int(TRAIN_HOURS * 3600.0 / secs_per_ep)
    EPOCHS = min(EPOCHS, cfg.EPOCHS)
    print(f"\nprobe: {secs_per_ep:.1f} s/epoch on {E.gpu_name}")
    print(f"TRAIN_HOURS={TRAIN_HOURS} -> {EPOCHS} epochs "
          f"(capped at cfg.EPOCHS={cfg.EPOCHS})")
    assert EPOCHS >= 2, "budget too small even for 2 epochs -- raise TRAIN_HOURS"
else:
    print(f"fixed schedule: {EPOCHS} epochs")

## The run

One arm: `alpha` starts at `ALPHA_PHYS` and is rebalanced on the projection
head every 200 steps. Everything else is §7.5 verbatim: AdamW, cosine decay to
`1e-5`, no warmup, no mixed precision, grad-clip 1.0. Checkpoints and the
history go to `checkpoints/full/` on the Volume as they are produced, so a
session that dies mid-run still leaves the best checkpoint so far.

In [ ]:
if have_hist:
    print("skipped -- history.json already on the Volume (FORCE_RETRAIN to redo)")
else:
    m = fno2d.build("primary")
    t0 = time.perf_counter()
    hist = training.train(
        m, str(paths["train"]), str(paths["val"]), out_dir=str(CKPT_DIR),
        device=DEV, epochs=EPOCHS, num_workers=WORKERS, alpha=cfg.ALPHA_PHYS,
        progress=tqdm)
    print(f"arm 'full' finished in {(time.perf_counter()-t0)/3600:.2f} h "
          f"({EPOCHS} epochs)")

In [ ]:
h = hist
fig, ax = plt.subplots(2, 3, figsize=(11.5, 6.0))
ep = np.arange(len(h["train"]))
ax[0, 0].semilogy(ep, [r["total"] for r in h["train"]], lw=1.0, label="full")
ax[0, 1].semilogy(ep, [r["rel_l2"] for r in h["val"]], lw=1.0, label="full")
ax[0, 2].semilogy(ep, [r["ring"] for r in h["val"]], lw=1.0, label="full")
ax[1, 0].semilogy(ep, [r["phase"] for r in h["val"]], lw=1.0, label="full")
ax[1, 1].semilogy(ep, [max(r["phys"], 1e-12) for r in h["train"]], lw=1.0,
                  label="full: L_phys")
ax[1, 2].semilogy(ep, [max(a, 1e-12) for a in h["alpha"]], lw=1.0, label="full")

ax[0, 0].set(xlabel="epoch", ylabel="train total loss", title="training loss")
ax[0, 1].set(xlabel="epoch", ylabel="val rel-L2", title="field error")
ax[0, 1].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[0, 2].set(xlabel="epoch", ylabel="val rel-L2 on the ring", title="receiver ring")
ax[1, 0].set(xlabel="epoch", ylabel="phase error (periods)", title="arrival error")
ax[1, 0].axhline(cfg.GATE_ARRIVAL_PERIODS, ls="--", c="C3", lw=1.0)
ax[1, 1].axhline(PHYS_FLOOR, ls=":", c="0.35", lw=1.2, label="floor (true field)")
ax[1, 1].set(xlabel="epoch", ylabel="L_phys", title="physics residual vs its floor")
ax[1, 2].set(xlabel="epoch", ylabel="alpha", title="balanced alpha")
for a_ in ax.ravel():
    a_.legend(fontsize=7)
fig.tight_layout()
savefig(fig, "03_training_curves.png")
plt.show()

## Gates

`rel_l2 < 5%` on the field and arrival error `< 0.05` periods, both from §11.3.
The arrival error is measured as receiver *phase* error divided by 2 pi, not by
synthesising a time trace: the band is 0.66-1.34 f_c, so a synthesised trace has
a time resolution of about 1.5 periods and could not resolve a 0.05-period
error even in principle.

In [ ]:
from src.data.dataset import WaveDataset as WD

va = WD(str(paths["val"]), train=False)
vl = make_loader(va, batch_size=4, shuffle=False, num_workers=0)

ck = CKPT_DIR / "best.pt"
m, meta = training.load(ck, device=DEV)
ev = training.evaluate(m, vl, DEV, per_freq=True)
print(f"--- full  (best epoch {meta['epoch']}, alpha {meta['alpha']})")
print(ev)
va.close()

evals = {"full": ev}

In [ ]:
# If the nophys arm was run headless earlier, its history is on the Volume;
# pick it up here so the ablation row of the thesis table is real rather than '-'.
import json as _json

for arm in ("nophys",):
    hp = E.checkpoints / arm / "best.pt"
    if hp.exists():
        va2 = WD(str(paths["val"]), train=False)
        vl2 = make_loader(va2, batch_size=4, shuffle=False, num_workers=0)
        m2, meta2 = training.load(hp, device=DEV)
        evals[arm] = training.evaluate(m2, vl2, DEV, per_freq=True)
        print(f"--- {arm}  (best epoch {meta2['epoch']})")
        print(evals[arm])
        va2.close()
    else:
        print(f"{arm}: not on the Volume -- ablation row will read '-' "
              f"(modal run modal_app.py::train_arm --arm {arm})")

In [ ]:
ck = CKPT_DIR / "best.pt"
m, meta = training.load(ck, device=DEV)
print("stored arch:", meta["arch"])
print("stored val :", meta["val"])
with torch.no_grad():
    p1 = m(x)
    training.save(m, E.checkpoints / "roundtrip.pt", epoch=meta["epoch"])
    m2, _ = training.load(E.checkpoints / "roundtrip.pt", device=DEV)
    p2 = m2(x)
print(f"\nround-trip max |difference|: {float((p1-p2).abs().max()):.3e}")
assert torch.equal(p1, p2), "checkpoint round-trip is not exact"
print("PASS  bit-identical after save/load")
(E.checkpoints / "roundtrip.pt").unlink()

In [ ]:
k_need = fno2d.band_in_modes()
rows = []
for v, kw in cfg.VARIANTS.items():
    mv = fno2d.build(v)
    rows.append((v, kw["d_v"], kw["kmax"], f"{mv.effective_params():,}",
                 f"{mv.allocated_params():,}",
                 "yes" if kw["kmax"] >= k_need else "NO -- truncates the band"))
    del mv
table(rows, ["variant", "d_v", "kmax", "effective", "allocated", "covers the band?"])
print(f"\nband_in_modes() = {k_need:.1f} at nu = {min(cfg.NU_LIST)}, "
      f"f = {max(cfg.FREQS):.2f} f_c")

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "mode": MODE,
    "epochs": len(hist.get("train", [])), "budget_hours": TRAIN_HOURS,
    "params": {v: fno2d.build(v).effective_params() for v in cfg.VARIANTS},
    "phys_floor_true_field": PHYS_FLOOR,
    "phys_zero_prediction": float(t_zero.phys),
    "alpha_balanced_projection": a_fresh,
    "alpha_balanced_all_params": a_all,
    "arms": {k: dict(rel_l2=v.rel_l2, ring=v.ring_rel_l2, phase=v.phase_periods,
                     per_freq=v.per_freq, gates=v.gates())
             for k, v in evals.items()},
}
dump(record, "03_train_fno.json")
print("\nCheckpoints:")
for p in sorted(E.checkpoints.glob("*/best.pt")):
    print(f"  {p}  {p.stat().st_size/1e6:.1f} MB")
print("\nStage D evaluates checkpoints/full/best.pt as a forward operator.")

---
# Stage D -- The surrogate as a forward operator  (§11.2 step 7's gates)

The gates measured on the test split, and the three figures §11.3 calls
non-negotiable for the forward model:

1. predicted vs true scattered wavefield, for a defect the network never saw;
2. predicted vs true at all 32 receivers -- the only part of the field the
   inversion reads;
3. error against `|k|`, which says *where* in the spectrum the remaining error
   lives.

The test split is the honest one. `generate` gives train and val the reduced
source pool `SRC_TRAIN`, and gives test **all** sources, so `SRC_HELDOUT =
(3, 6)` appears in test and nowhere else. Every number below is therefore
reported twice: over trained illuminations, and over the two the network has
never been shown. A surrogate that has memorised eight source patterns rather
than learned an operator separates cleanly on that split, and no amount of
held-out *geometry* would reveal it.

Read-only: no training, no solving. Runs in a couple of minutes.

In [ ]:
from src import features as feat
from src import losses as L
from src import training
from src.data.dataset import WaveDataset, batch_to_model, make_loader, to_device
from src.models.fno2d import band_in_modes
from src.solver import harmonic as H

assert paths["test"].exists(), "stage B failed -- no test split"
assert CKPT.exists(), f"no checkpoint at {CKPT} -- stage C must run first"

model, meta = training.load(CKPT, device=DEV)
model.eval()
print(f"loaded {CKPT}")
print(f"  arch  {meta['arch']}")
print(f"  epoch {meta['epoch']}   val {meta['val']}   alpha {meta['alpha']}")
print(f"  {model.effective_params():,} effective parameters")

## Gates on the test split

`evaluate` puts the dataset in eval mode, which turns off the frequency
subsetting: every sample contributes all `M_FREQ = 20` lines, so a batch of 4
samples is 80 rows through the network. The per-frequency breakdown comes from
the same pass.

In [ ]:
ts = WaveDataset(str(paths["test"]), train=False)
tl = make_loader(ts, batch_size=4, shuffle=False, num_workers=0)
print(f"test split: {len(ts)} samples, {ts.n_freq} frequencies each, "
      f"{len(ts) * ts.n_freq} rows")

t0 = time.perf_counter()
ev = training.evaluate(model, tl, DEV, per_freq=True)
print(f"\nevaluated in {time.perf_counter()-t0:.1f} s\n")
print(ev)

### Per-sample errors, split by illumination

`evaluate` returns means. The means are what the gates are stated on, but they
hide the tail, and the tail is what the inversion trips over: one sample at
30% error is a failed inversion even if the mean is 3%. The loop below
recomputes the same two metrics per sample so the distribution, the worst
cases and the held-out-source comparison are all available.

In [ ]:
recv = training.receivers_tensor(DEV)


@torch.no_grad()
def per_sample(loader):
    rows = []
    for batch in loader:
        b = to_device(batch, DEV)
        x, y = batch_to_model(b)
        p = model(x)
        nf = b["freqs"].shape[1]
        d = (p - y).reshape(-1, nf, *y.shape[1:])
        t = y.reshape(-1, nf, *y.shape[1:])
        dims = (1, 2, 3, 4)
        f_err = (d.pow(2).sum(dims).sqrt() / t.pow(2).sum(dims).sqrt()).cpu()
        ry, rx = recv[:, 0], recv[:, 1]
        dr, tr = d[..., ry, rx], t[..., ry, rx]
        r_err = (dr.pow(2).sum(dims).sqrt() / tr.pow(2).sum(dims).sqrt()).cpu()
        for k in range(f_err.shape[0]):
            rows.append(dict(index=int(b["index"][k]), src=int(b["src_idx"][k]),
                             nu_idx=int(b["nu_idx"][k]),
                             theta=_np(b["theta"][k]).tolist(),
                             field=float(f_err[k]), ring=float(r_err[k])))
    return rows


ps = per_sample(tl)
fe = np.array([r["field"] for r in ps])
re_ = np.array([r["ring"] for r in ps])
src = np.array([r["src"] for r in ps])
held = np.isin(src, cfg.SRC_HELDOUT)

table([("all", f"{len(fe)}", f"{fe.mean():.4f}", f"{np.median(fe):.4f}",
        f"{np.percentile(fe, 95):.4f}", f"{fe.max():.4f}", f"{re_.mean():.4f}"),
       ("trained sources", f"{(~held).sum()}", f"{fe[~held].mean():.4f}",
        f"{np.median(fe[~held]):.4f}", f"{np.percentile(fe[~held], 95):.4f}",
        f"{fe[~held].max():.4f}", f"{re_[~held].mean():.4f}"),
       (f"held out {cfg.SRC_HELDOUT}", f"{held.sum()}", f"{fe[held].mean():.4f}",
        f"{np.median(fe[held]):.4f}", f"{np.percentile(fe[held], 95):.4f}",
        f"{fe[held].max():.4f}", f"{re_[held].mean():.4f}")],
      ["subset", "n", "mean", "median", "p95", "max", "ring mean"])

pen = fe[held].mean() / max(fe[~held].mean(), 1e-30) - 1.0
print(f"\nheld-out illumination costs {pen:+.1%} on the mean field error")
print(f"gate rel-L2 < {cfg.GATE_REL_L2:.0%}: "
      f"{'PASS' if fe[held].mean() < cfg.GATE_REL_L2 else 'FAIL'} even on held-out sources"
      if held.any() else "")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11.2, 3.0))

bins = np.linspace(0, max(fe.max(), cfg.GATE_REL_L2 * 1.5), 40)
ax[0].hist(fe[~held], bins=bins, alpha=0.7, label="trained sources", density=True)
if held.any():
    ax[0].hist(fe[held], bins=bins, alpha=0.7, label=f"held out {cfg.SRC_HELDOUT}",
               density=True)
ax[0].axvline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0, label=f"gate {cfg.GATE_REL_L2:.0%}")
ax[0].set(xlabel="field rel-L2", ylabel="density", title="per-sample field error")
ax[0].legend(fontsize=7.5)

R = np.array([r["theta"][2] for r in ps])
lam = np.array([cfg.cs_over_cp(cfg.NU_LIST[r["nu_idx"]]) / cfg.FC for r in ps])
ax[1].plot(R / lam, fe, ".", ms=3, alpha=0.5)
ax[1].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[1].axvline(cfg.R_MIN_LS, ls=":", c="0.4", lw=1.0)
ax[1].set(xlabel="R / lambda_s", ylabel="field rel-L2",
          title="error vs defect size\n(small voids scatter least, so relative\n"
                "error is hardest there)")

sx = np.array([cfg.SOURCE_XY[r["src"]][0] for r in ps])
sy = np.array([cfg.SOURCE_XY[r["src"]][1] for r in ps])
dist = np.hypot(np.array([r["theta"][0] for r in ps]) - sx,
                np.array([r["theta"][1] for r in ps]) - sy) / lam
ax[2].plot(dist, fe, ".", ms=3, alpha=0.5)
ax[2].axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0)
ax[2].set(xlabel="source-defect distance / lambda_s", ylabel="field rel-L2",
          title="error vs standoff")
fig.tight_layout()
savefig(fig, "04_error_distribution.png")
plt.show()

## Figure 1 -- the predicted wavefield

One test sample, at the bottom, middle and top of the band. `|u_s|` for truth
and prediction share a colour scale per row; the third column is
`|u_s_pred - u_s_true|` on the *same* scale, so a visible error panel is a real
error and not a rescaled one. The void outline is the `chi = 0.5` contour of the
same soft indicator the network was given as input, and the source is marked.

The sample shown is the **worst held-out-source case** by field error, because a
montage of a median case is a picture of the network working and a montage of
the worst case is the only one that can show *how* it fails. If the error
concentrates on the void boundary, that is the interface width; if it trails
behind the scattered front, that is dispersion; if it sits at the domain edge,
that is the absorber leaking into the labels.

In [ ]:
pool = np.where(held)[0] if held.any() else np.arange(len(ps))
pick = int(pool[np.argmax(fe[pool])])
rec = ps[pick]
print(f"sample {rec['index']}  src {rec['src']} "
      f"{'(HELD OUT)' if rec['src'] in cfg.SRC_HELDOUT else ''}  "
      f"nu {cfg.NU_LIST[rec['nu_idx']]}  theta {np.round(rec['theta'], 4)}  "
      f"field rel-L2 {rec['field']:.4f}")

one = WaveDataset(str(paths["test"]), train=False)
b = to_device({k: (v.unsqueeze(0) if torch.is_tensor(v) else v)
               for k, v in one[rec["index"]].items()}, DEV)
x1, y1 = batch_to_model(b)
with torch.no_grad():
    p1 = model(x1)
zt = _np(feat.channels_to_complex(y1))          # [M, 2, ny, nx] complex
zp = _np(feat.channels_to_complex(p1))
chi1 = _np(b["chi"][0])
sy_, sx_ = cfg.SOURCES_NET[rec["src"]]
print(f"phasor stack {zt.shape}")

In [ ]:
show_m = [0, cfg.M_FREQ // 2, cfg.M_FREQ - 1]
fig, ax = plt.subplots(len(show_m), 4, figsize=(11.6, 2.65 * len(show_m)))
yx = np.arange(cfg.N_NET)

for r_, m in enumerate(show_m):
    at = np.abs(zt[m]).sum(0) ** 0.5      # sqrt of summed |u|^2 over components
    ap = np.abs(zp[m]).sum(0) ** 0.5
    err = np.abs(zp[m] - zt[m]).sum(0) ** 0.5
    vmax = float(at.max())
    for c_, (img, ttl) in enumerate([(at, "true |u_s|"), (ap, "predicted"),
                                    (err, "|error|")]):
        im = ax[r_, c_].imshow(img, origin="lower", cmap="magma", vmin=0, vmax=vmax)
        ax[r_, c_].contour(yx, yx, chi1, levels=[0.5], colors="c", linewidths=0.9)
        ax[r_, c_].plot(sx_, sy_, "w*", ms=7)
        ax[r_, c_].set(xticks=[], yticks=[])
        ax[r_, c_].grid(False)
        if r_ == 0:
            ax[r_, c_].set_title(ttl, fontsize=9)
    ax[r_, 0].set_ylabel(f"f = {cfg.FREQS[m]:.3f} f_c", fontsize=8.5)
    fig.colorbar(im, ax=ax[r_, 2], fraction=0.046)

    ph = np.angle(zp[m, 0] * np.conj(zt[m, 0]))
    mask = np.abs(zt[m, 0]) < 0.05 * np.abs(zt[m, 0]).max()
    ph = np.where(mask, np.nan, ph)
    im2 = ax[r_, 3].imshow(ph, origin="lower", cmap="twilight_shifted",
                           vmin=-np.pi, vmax=np.pi)
    ax[r_, 3].contour(yx, yx, chi1, levels=[0.5], colors="k", linewidths=0.9)
    ax[r_, 3].set(xticks=[], yticks=[])
    ax[r_, 3].grid(False)
    if r_ == 0:
        ax[r_, 3].set_title("phase error, u_x\n(blank where |u| < 5% of max)",
                            fontsize=9)
    fig.colorbar(im2, ax=ax[r_, 3], fraction=0.046)

fig.suptitle(f"worst held-out-source sample {rec['index']}: "
             f"rel-L2 {rec['field']:.3f}", fontsize=10)
fig.tight_layout()
savefig(fig, "04_fig1_wavefield.png")
plt.show()

### Time-domain context

The dataset keeps 64 downsampled velocity snapshots for `N_VIS_SAMPLES = 8`
samples per split. These are the **total** field from the solver, not the
network's output -- the network predicts phasors and has no time axis. They are
here because the phasor panels above are hard to read without knowing what the
wave was doing: the frames show the incident front crossing the void, the
scattered wave leaving it, and both being absorbed at the walls.

In [ ]:
import h5py

with h5py.File(paths["test"], "r") as f:
    vis_idx = f["vis/index"][:] if "vis/index" in f else np.array([], dtype=int)
    if len(vis_idx):
        k = int(np.argmin(np.abs(vis_idx - rec["index"])))
        fr = f["vis/frames"][k]                     # [n_frames, 2, ny, nx]
        vis_sample = int(vis_idx[k])
    else:
        fr = None

if fr is None:
    print("no visualisation frames in this file")
else:
    print(f"frames for sample {vis_sample} "
          f"({'the montage sample' if vis_sample == rec['index'] else 'the nearest one'})"
          f": {fr.shape}, every {cfg.SAVE_EVERY} steps")
    ks = np.linspace(fr.shape[0] * 0.18, fr.shape[0] - 1, 5).astype(int)
    v = np.abs(fr[:, 0]).max()
    fig, ax = plt.subplots(1, len(ks), figsize=(2.15 * len(ks), 2.4))
    for a_, kk in zip(ax, ks):
        a_.imshow(fr[kk, 0], origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
        a_.set(xticks=[], yticks=[],
               title=f"t = {kk * cfg.SAVE_EVERY * cfg.DT:.1f} T_p")
        a_.grid(False)
    fig.suptitle("total v_x, solver frames (not a network output)", fontsize=9)
    fig.tight_layout()
    savefig(fig, "04_frames.png")
    plt.show()

## Figure 2 -- all 32 receivers

The ring is 32 pixels out of 16384, and it is the entire input to the inversion.
§11.3 asks for predicted-vs-true A-scans at all 32 receivers; what the network
produces is a phasor, so this comes in two panels and the distinction between
them is worth being explicit about.

**Left: the frequency-domain gather.** Real and imaginary parts against receiver
index, at three frequencies, in physical units (the stored values are divided
by the incident domain maximum, so they are multiplied back). This is the
actual predicted quantity, compared without any further processing. The phase
of these numbers is what the inversion minimises.

**Right: a band-limited time reconstruction.**
`u(t) = 2 df Re sum_m u_hat_m exp(i omega_m t)` over the 20 lines, applied
*identically* to prediction and truth. It is not the solver's A-scan: 20 lines
spanning 0.66-1.34 f_c give a time resolution of about 1.5 periods, and the
reconstruction has no information outside the band. No attempt is made to
recover the absolute time origin. It is included because agreement here is
easier to *see* than agreement in a scatter of complex numbers -- and because a
phase error that the gather hides as a small rotation shows up in the time trace
as a shifted arrival, which is the failure the inversion cares about.

In [ ]:
scale = _np(b["scale"][0])                      # [M], the per-line divisor
ry, rx = _np(recv[:, 0]), _np(recv[:, 1])
gt = zt[:, :, ry, rx] * scale[:, None, None]    # [M, 2, R] physical
gp = zp[:, :, ry, rx] * scale[:, None, None]

fig = plt.figure(figsize=(11.6, 6.2))
gs = fig.add_gridspec(3, 2, width_ratios=[1.0, 1.15])

for r_, m in enumerate(show_m):
    a_ = fig.add_subplot(gs[r_, 0])
    ridx = np.arange(len(ry))
    a_.plot(ridx, gt[m, 0].real, "-", lw=1.2, c="C0", label="true Re u_x")
    a_.plot(ridx, gp[m, 0].real, "--", lw=1.2, c="C1", label="pred Re u_x")
    a_.plot(ridx, gt[m, 0].imag, "-", lw=1.0, c="C2", alpha=0.8, label="true Im u_x")
    a_.plot(ridx, gp[m, 0].imag, "--", lw=1.0, c="C3", alpha=0.8, label="pred Im u_x")
    a_.set_ylabel(f"f = {cfg.FREQS[m]:.3f}", fontsize=8.5)
    if r_ == 0:
        a_.legend(fontsize=6.5, ncol=2)
        a_.set_title("frequency-domain gather, 32 receivers", fontsize=9)
    if r_ == len(show_m) - 1:
        a_.set_xlabel("receiver index (counter-clockwise around the ring)")

om_ = 2.0 * np.pi * np.asarray(cfg.FREQS)
tt = np.linspace(0.0, 12.0, 900)
kern = np.exp(1j * om_[:, None] * tt[None, :])
ut = 2.0 * cfg.DF * np.real(np.tensordot(gt[:, 0, :], kern, axes=(0, 0)))   # [R, T]
up = 2.0 * cfg.DF * np.real(np.tensordot(gp[:, 0, :], kern, axes=(0, 0)))

a_ = fig.add_subplot(gs[:, 1])
step = 1.05 * np.abs(ut).max()
for i in range(ut.shape[0]):
    a_.plot(tt, ut[i] + i * step, "-", lw=0.7, c="C0")
    a_.plot(tt, up[i] + i * step, "--", lw=0.7, c="C1")
a_.plot([], [], "-", c="C0", label="true")
a_.plot([], [], "--", c="C1", label="predicted")
a_.set(xlabel="t (arbitrary origin) / T_p", yticks=[],
       title="band-limited reconstruction, u_x at all 32 receivers\n"
             "(20 lines over 0.66-1.34 f_c: resolution ~1.5 periods)")
a_.legend(fontsize=8, loc="upper right")

fig.tight_layout()
savefig(fig, "04_fig2_receivers.png")
plt.show()

num = np.abs(gp - gt).ravel()
den = np.abs(gt).ravel()
print(f"ring rel-L2 for this sample: "
      f"{np.linalg.norm(num)/np.linalg.norm(den):.4f}")
print(f"ring amplitude is {np.abs(gt).max()/np.abs(zt*scale[:,None,None,None]).max():.4f} "
      f"of the domain maximum -- the two normalisation scales of §5.3 differ by about "
      f"two orders of magnitude, which is why the receiver misfit uses scale_recv "
      f"and the network inputs use scale")

## Figure 3 -- where the error lives in `|k|`

The error field's power, binned by integer radius `|k|` on the `128^2` grid,
against the true field's, both from `rfft2` with `norm='ortho'`. Three vertical
lines matter:

- `band_in_modes()`: the mode index the shortest shear wave in the band
  occupies. Below this is physics the network must represent.
- `KMAX = 28`: the truncation. Above it the spectral layers contribute
  nothing, and whatever the network gets right up there comes from the pointwise
  `1x1` convolutions alone.
- `K_NYQUIST`: the grid's own limit.

The expected picture: relative error roughly flat and small below
`band_in_modes`, rising between there and `KMAX`, and the truncated region
carrying little true power -- which is the justification for truncating at all.
What would be a problem is significant *true* power above `KMAX`, because that
is signal the architecture has thrown away, and no amount of training recovers
it. That is also the prediction the `tiny` variant (`KMAX = 16`, inside the
band) is there to test.

In [ ]:
def radial_power(z):
    # [.., 2, ny, nx] complex field -> (k, mean power per integer |k| bin).
    t = torch.from_numpy(np.ascontiguousarray(z))
    F_ = torch.fft.rfft2(t.real, norm="ortho") + 1j * torch.fft.rfft2(t.imag,
                                                                      norm="ortho")
    p = _np(F_.abs().pow(2).sum(tuple(range(F_.dim() - 2))))     # [ny, nkx]
    ny, nkx = p.shape
    fy = np.fft.fftfreq(ny) * ny
    fx = np.arange(nkx)
    kk = np.hypot(fy[:, None], fx[None, :])
    kb = np.rint(kk).astype(int)
    nb = kb.max() + 1
    tot = np.bincount(kb.ravel(), weights=p.ravel(), minlength=nb)
    cnt = np.bincount(kb.ravel(), minlength=nb).clip(1)
    return np.arange(nb), tot / cnt


k_t, p_t = radial_power(zt)
k_e, p_e = radial_power(zp - zt)
k_need = band_in_modes()

fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.3))
ax[0].semilogy(k_t, np.maximum(p_t, 1e-24), lw=1.2, label="true field")
ax[0].semilogy(k_e, np.maximum(p_e, 1e-24), lw=1.2, label="error")
ax[1].semilogy(k_t, np.sqrt(np.maximum(p_e, 1e-30) / np.maximum(p_t, 1e-30)),
               lw=1.2, c="C3")
for a_ in ax:
    a_.axvline(k_need, ls=":", c="C2", lw=1.2,
               label=f"band top needs |k| = {k_need:.1f}")
    a_.axvline(cfg.KMAX, ls="--", c="0.3", lw=1.2, label=f"KMAX = {cfg.KMAX}")
    a_.axvline(cfg.K_NYQUIST, ls="-.", c="0.6", lw=1.0,
                label=f"Nyquist = {cfg.K_NYQUIST}")
    a_.set_xlabel("|k| (integer modes on the 128^2 grid)")
ax[0].set(ylabel="mean power per mode", title="spectra, all 20 lines")
ax[0].legend(fontsize=7)
ax[1].set(ylabel="relative error amplitude", title="error / signal vs |k|")
ax[1].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "04_fig3_error_spectrum.png")
plt.show()

above = p_t[cfg.KMAX + 1:].sum() / max(p_t.sum(), 1e-30)
inband = np.sqrt(p_e[:int(k_need) + 1].sum() / max(p_t[:int(k_need) + 1].sum(), 1e-30))
print(f"true power above KMAX (thrown away by truncation): {above:.3e}")
print(f"relative error below |k| = {k_need:.0f} (the physical band): {inband:.4f}")
print(f"relative error over all |k|                       : "
      f"{np.sqrt(p_e.sum()/max(p_t.sum(),1e-30)):.4f}")

## Per-frequency error

`evaluate(..., per_freq=True)` accumulates the numerator and denominator
separately per line, so this is a proper relative error per frequency and not an
average of ratios. The top of the band has the fewest points per wavelength on
both grids, so it should be the worst line -- the same ordering check 5 in stage
A found in the solver's own convergence. If the *bottom* of the band is worst,
something is wrong with the deconvolution rather than with the network.

In [ ]:
pf = np.asarray(ev.per_freq)
fig, ax = plt.subplots(figsize=(6.6, 3.1))
ax.bar(cfg.FREQS[:len(pf)], pf, width=0.85 * cfg.DF, color="C0", alpha=0.85)
ax.axhline(cfg.GATE_REL_L2, ls="--", c="C3", lw=1.0, label=f"gate {cfg.GATE_REL_L2:.0%}")
ax.axhline(pf.mean(), ls=":", c="0.35", lw=1.0, label=f"mean {pf.mean():.3f}")
ax.set(xlabel="f / f_c", ylabel="rel-L2", title="test error per frequency line")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "04_per_frequency.png")
plt.show()

table([(f"{m}", f"{cfg.FREQS[m]:.4f}", f"{pf[m]:.4f}",
        "worst" if m == int(pf.argmax()) else "")
       for m in range(len(pf))], ["m", "f / f_c", "rel-L2", ""])

## Throughput

The point of the surrogate is that the inversion can afford tens of thousands
of forward evaluations. This measures what one costs, and compares it against
the solver throughput stage B measured on *this* GPU -- so the speedup quoted is
measured rather than asserted.

Stage 1 alone evaluates `SCREEN_GRID^2 = 256` candidates over 6 frequencies, and
stages 2 and 3 add a few thousand more. At solver cost that is weeks per
inversion.

In [ ]:
with torch.no_grad():
    for _ in range(3):
        model(x1)
    if DEV.startswith("cuda"):
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    n_rep = 20
    for _ in range(n_rep):
        model(x1)
    if DEV.startswith("cuda"):
        torch.cuda.synchronize()
    per_call = (time.perf_counter() - t0) / n_rep

rows_per_call = x1.shape[0]
print(f"{rows_per_call} rows (1 geometry x {cfg.M_FREQ} lines) per call")
print(f"  {per_call*1e3:8.2f} ms per call")
print(f"  {per_call/rows_per_call*1e3:8.3f} ms per (geometry, frequency)")
print(f"  {rows_per_call/per_call:8.0f} rows/s")

solver_s = None
gp_ = E.results / "02_dataset_generation.json"
if gp_.exists():
    thr_ = json.loads(gp_.read_text()).get("throughput_cell_steps_per_s")
    if thr_:
        # one geometry is N_FINE_TOTAL^2 cells x NT steps, and yields all M_FREQ
        # lines at once because the DFT runs inside the time loop
        solver_s = cfg.N_FINE_TOTAL ** 2 * cfg.NT / thr_
if solver_s:
    n_cand = cfg.SCREEN_GRID ** 2
    print(f"\nsolver: {solver_s:.2f} s per geometry (all {cfg.M_FREQ} lines), from "
          f"stage B's measured throughput")
    print(f"speedup: {solver_s/per_call:,.0f}x per forward evaluation")
    print(f"stage 1's {n_cand} candidates: {n_cand*per_call:.2f} s surrogate "
          f"vs {n_cand*solver_s/3600:.1f} h solver")
else:
    print("\n(no throughput in 02_dataset_generation.json -- stage B was skipped)")

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "checkpoint": str(CKPT),
    "arch": meta["arch"], "epoch": meta["epoch"],
    "gates": ev.gates(),
    "test": dict(rel_l2=ev.rel_l2, ring=ev.ring_rel_l2, phase=ev.phase_periods,
                 per_freq=ev.per_freq),
    "per_sample": dict(
        n=len(fe), mean=float(fe.mean()), median=float(np.median(fe)),
        p95=float(np.percentile(fe, 95)), max=float(fe.max()),
        trained_mean=float(fe[~held].mean()) if (~held).any() else None,
        heldout_mean=float(fe[held].mean()) if held.any() else None,
        heldout_penalty=float(pen) if held.any() else None),
    "spectrum": dict(true_power_above_kmax=float(above),
                     rel_error_in_band=float(inband),
                     kmax=cfg.KMAX, band_in_modes=float(k_need)),
    "throughput": dict(seconds_per_call=per_call, rows_per_call=int(rows_per_call),
                       solver_seconds_per_sample=solver_s),
    "worst_heldout_sample": rec,
}
dump(record, "04_forward_eval.json")

ts.close()
one.close()
print()
for k, v in ev.gates().items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
if all(ev.gates().values()):
    print("\nThe forward operator is good enough to invert against.  Stage E.")
else:
    print("\nSTOP.  §11.3: the forward gates come before the inverse ones.  An\n"
          "inversion against a surrogate that is 10% wrong at the receivers will\n"
          "converge confidently to the wrong defect, and the misfit at the answer\n"
          "will look plausible because the model error is systematic.")

---
# Stage E -- Inversion  (§11.2 steps 9-11)

- **step 9**, the gradient check. Three ways of computing the same derivative,
  agreeing to `GATE_GRAD_SIGFIGS = 3` significant figures in double precision.
- **step 10**, one noiseless inversion, end to end, with every stage's trace,
  and the misfit landscape figure §11.3 calls non-negotiable.
- **step 11**, the success-rate statistic at 30 dB over several cases, split by
  whether the illumination was held out of training, plus an SNR sweep.

There is also a cycle-skipping demonstration, which is not on the checklist but
is the reason the pipeline has four stages instead of one.

**Runtime.** Minutes for steps 9 and 10; the statistics scale with `N_CASES`.
`QUICK = True` (first cell) keeps them short; `QUICK = False` runs the full
§11.2 sizes and adds roughly 1-2 h.

In [ ]:
N_CASES = 12 if QUICK else 40       # step 11, at 30 dB
N_SNR_CASES = 6 if QUICK else 16    # per SNR in the sweep
MAP_N = 31 if QUICK else 41         # misfit map resolution
print(f"{'QUICK' if QUICK else 'FULL'}: {N_CASES} cases at 30 dB, "
      f"{N_SNR_CASES} per SNR over {cfg.SNR_DB_SWEEP}, {MAP_N}x{MAP_N} misfit maps")

In [ ]:
import copy

import h5py

from src import training
from src.data.dataset import load_incident, load_inversion_case
from src.geometry.sdf import Circle, net_coords, soft_indicator
from src.inverse import invert as INV
from src.inverse.misfit import (InverseCase, Objective, SurrogateForward,
                                basin_width, misfit_map)
from src.models.fno2d import to_double

assert paths["test"].exists() and CKPT.exists(), "stages B and C must run first"

model, meta = training.load(CKPT, device=DEV)
inc = load_incident(str(paths["test"]), device=DEV)
fwd = SurrogateForward(model, inc, device=DEV)
family = Circle()

print(f"model {meta['arch']}, epoch {meta['epoch']}")
print(f"incident cache: phasors {tuple(inc['phasors'].shape)}  "
      f"scale {tuple(inc['scale'].shape)}  scale_recv {tuple(inc['scale_recv'].shape)}")
print(f"forward frozen: {not any(p.requires_grad for p in fwd.model.parameters())}")

with h5py.File(paths["test"], "r") as f:
    src_all = f["samples/src_idx"][:]
    theta_all = f["samples/theta"][:]
    n_test = int(f.attrs["n_samples"])
    held_mask = np.isin(src_all, cfg.SRC_HELDOUT)
print(f"\ntest split: {n_test} samples, {held_mask.sum()} on held-out sources "
      f"{cfg.SRC_HELDOUT}")

## Step 9a -- the geometry derivative

`d chi / d theta` for a circle, three ways: the closed form of §8.2, autodiff
through `sigmoid(-phi/eps)`, and central differences. The closed form is

    d chi / d xc = sigma' (1/eps) cos(alpha)
    d chi / d yc = sigma' (1/eps) sin(alpha)
    d chi / d R  = sigma' (1/eps)

with `sigma'` sharply peaked on the boundary. Two things are worth seeing rather
than asserting. The support is an annulus of width about `eps`, so **all**
sensitivity to the geometry lives within a couple of cells of the boundary --
which is why the interface width is annealed rather than fixed, and why the
position estimate cannot be sharper than the interface it is estimating. And
the three derivatives have distinct angular signatures: the radius derivative is
uniform round the annulus (a breathing monopole), the position derivatives
carry `cos` and `sin` (dipoles). Three parameters, three orthogonal modes --
which is the reason all three are separately identifiable from one ring of data.

This is checked before the network is involved at all. If the geometry
derivative is wrong, every gradient in the pipeline is wrong, and no amount of
inspecting the optimiser would say so.

In [ ]:
yy, xx = net_coords(device=DEV, dtype=torch.float64)
eps_len = cfg.EPS_INTERFACE_CELLS * cfg.DX_NET
i0 = int(np.where(held_mask)[0][0]) if held_mask.any() else 0
theta0 = torch.tensor(theta_all[i0], dtype=torch.float64, device=DEV).unsqueeze(0)
print(f"at theta = {np.round(_np(theta0)[0], 5)}  (test sample {i0}, "
      f"source {src_all[i0]})")


def chi_of(t):
    return soft_indicator(family.sdf(t, yy, xx), eps_len)


ana = family.dchi_dtheta_analytic(theta0, yy, xx, eps_len)[0]       # [3, ny, nx]

# autodiff, as a VJP against a fixed random probe: <v, dchi/dtheta_i>
g = torch.Generator(device="cpu").manual_seed(0)
v = torch.randn(cfg.N_NET, cfg.N_NET, generator=g,
                dtype=torch.float64).to(DEV)
t_ad = theta0.clone().requires_grad_(True)
(chi_of(t_ad) * v).sum().backward()
vjp_ad = _np(t_ad.grad[0])
vjp_ana = _np((ana * v).sum(dim=(-2, -1)))

rows = []
for i, nm in enumerate(family.param_names):
    with torch.no_grad():
        h = 1e-7
        e = torch.zeros_like(theta0)
        e[0, i] = h
        fd = (chi_of(theta0 + e) - chi_of(theta0 - e)) / (2 * h)
        fd_v = float((fd[0] * v).sum())
    a_, b_, c_ = float(vjp_ana[i]), float(vjp_ad[i]), fd_v
    dig = lambda p, q: -math.log10(max(abs(p - q) / max(abs(p), abs(q), 1e-300), 1e-300))
    rows.append((nm, f"{a_:+.8e}", f"{b_:+.8e}", f"{c_:+.8e}",
                 f"{dig(a_, b_):.1f}", f"{dig(a_, c_):.1f}"))
table(rows, ["param", "analytic", "autodiff", "central FD",
             "digits ana/ad", "digits ana/FD"])

In [ ]:
a_np = _np(ana)
chi_np = _np(chi_of(theta0)[0])
fig, ax = plt.subplots(1, 4, figsize=(12.0, 2.9))
for i, nm in enumerate(family.param_names):
    m = np.abs(a_np[i]).max()
    im = ax[i].imshow(a_np[i], origin="lower", cmap="RdBu_r", vmin=-m, vmax=m)
    ax[i].contour(np.arange(cfg.N_NET), np.arange(cfg.N_NET), chi_np,
                  levels=[0.5], colors="k", linewidths=0.8)
    ax[i].set(title=f"d chi / d {nm}", xticks=[], yticks=[])
    ax[i].grid(False)
    fig.colorbar(im, ax=ax[i], fraction=0.046)

c_ = _np(theta0)[0]
r_pix = _np(torch.sqrt((xx - c_[0]) ** 2 + (yy - c_[1]) ** 2))
ax[3].plot(((r_pix - c_[2]) / cfg.DX_NET).ravel(), a_np[2].ravel(), ".", ms=1.5,
           alpha=0.35)
ax[3].axvline(-cfg.EPS_INTERFACE_CELLS, ls="--", c="C3", lw=1.0)
ax[3].axvline(cfg.EPS_INTERFACE_CELLS, ls="--", c="C3", lw=1.0,
              label=f"+/- eps = {cfg.EPS_INTERFACE_CELLS} cells")
ax[3].set(xlim=(-8, 8), xlabel="distance from boundary (cells)",
          ylabel="d chi / d R", title="all sensitivity is in the annulus")
ax[3].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_dchi_dtheta.png")
plt.show()

## Step 9b -- `dJ/dtheta` through the surrogate

Autodiff against central differences of the objective itself, in **double
precision** and **away from the minimum**.

Both of those are load-bearing. In float32 the field carries about seven
digits, and a central difference of a relative misfit built from that field
loses roughly half of them before the ratio is formed, leaving less than the
three significant figures the gate asserts -- so the check would fail for
precision reasons and get quietly disabled. `to_double` exists because
`nn.Module.double()` skips the complex spectral weights. And at `theta_true`
the gradient is approximately zero, where a relative comparison of two small
numbers reports noise; the check is done at `theta_true + delta` with `delta`
about a fifth of a shear wavelength, where the gradient is O(1) and both methods
have something to agree about.

The step size is swept rather than guessed. Central differences have error
`~ h^2 f''' + eps_machine f / h`, so the agreement is a V in `h`:
truncation-limited on the right, round-off-limited on the left. The reported
number is the bottom of the V. A monotone curve with no minimum would mean the
objective is not smooth at this point, which would be a finding rather than a
passing check.

In [ ]:
case0 = InverseCase.from_dict(load_inversion_case(str(paths["test"]), i0)).to(DEV)
case64 = InverseCase(d_obs=case0.d_obs.to(torch.complex128), src_idx=case0.src_idx,
                     nu_idx=case0.nu_idx,
                     theta_true=case0.theta_true.to(torch.float64))
fwd64 = SurrogateForward(to_double(copy.deepcopy(model)), inc, device=DEV,
                         dtype=torch.float64)
obj64 = Objective(fwd64, case64, family, band=cfg.BAND_STAGE3,
                  eps_cells=cfg.EPS_INVERT_END)

lam_s = case64.lambda_s
delta = torch.tensor([0.2 * lam_s, -0.15 * lam_s, 0.1 * lam_s],
                     dtype=torch.float64, device=DEV)
theta_g = (case64.theta_true.to(DEV) + delta).unsqueeze(0)
print(f"theta_true = {np.round(_np(case64.theta_true), 5)}")
print(f"theta_test = {np.round(_np(theta_g)[0], 5)}   "
      f"(offset {float(delta[:2].norm())/lam_s:.2f} lambda_s in position)")

t_ad = theta_g.clone().requires_grad_(True)
j0 = obj64.residual(t_ad)
j0.backward()
grad_ad = _np(t_ad.grad[0])
print(f"\nJ = {float(j0):.10e}")
print(f"autodiff dJ/dtheta = {np.array2string(grad_ad, precision=8)}")

In [ ]:
def fd_grad(h):
    out = np.zeros(3)
    with torch.no_grad():
        for i in range(3):
            e = torch.zeros_like(theta_g)
            e[0, i] = h
            out[i] = float(obj64.residual(theta_g + e)
                           - obj64.residual(theta_g - e)) / (2 * h)
    return out


hs = np.array([1e-2, 3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6, 3e-7]) * lam_s
digs = []
for h in hs:
    gfd = fd_grad(float(h))
    rel = np.abs(gfd - grad_ad) / np.maximum(np.abs(grad_ad), 1e-300)
    digs.append(-np.log10(np.maximum(rel, 1e-300)))
digs = np.array(digs)                      # [n_h, 3]

best_i = int(digs.min(axis=1).argmax())
best_h = float(hs[best_i])
best = fd_grad(best_h)
worst_digits = float(digs[best_i].min())

table([(nm, f"{grad_ad[i]:+.8e}", f"{best[i]:+.8e}", f"{digs[best_i, i]:.2f}")
       for i, nm in enumerate(family.param_names)],
      ["param", "autodiff", f"central FD (h = {best_h:.2e})", "digits"])

print(f"\nworst component agrees to {worst_digits:.2f} significant figures")
print(f"gate GATE_GRAD_SIGFIGS = {cfg.GATE_GRAD_SIGFIGS}: "
      f"{'PASS' if worst_digits >= cfg.GATE_GRAD_SIGFIGS else 'FAIL'}")
assert worst_digits >= cfg.GATE_GRAD_SIGFIGS, (
    "gradient check failed -- do not run an inversion against this gradient")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.2))
for i, nm in enumerate(family.param_names):
    ax.plot(hs / lam_s, digs[:, i], "o-", ms=3.5, lw=1.0, label=nm)
ax.axhline(cfg.GATE_GRAD_SIGFIGS, ls="--", c="C3", lw=1.0,
           label=f"gate {cfg.GATE_GRAD_SIGFIGS} digits")
ax.axvline(best_h / lam_s, ls=":", c="0.4", lw=1.0, label="best h")
ax.set(xscale="log", xlabel="h / lambda_s", ylabel="digits of agreement",
       title="central-difference step sweep\n"
             "(right: truncation error;  left: round-off)")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_gradient_check.png")
plt.show()
del fwd64, obj64
if DEV.startswith("cuda"):
    torch.cuda.empty_cache()

## Step 10 -- one inversion, noiseless

Four stages, and each exists because the one before it cannot do its job:

| stage | what it does | band | why |
|-------|--------------|------|-----|
| 0 | RingCNN guess (stage F) | - | free, and right when the defect is in-distribution |
| 1 | 256-candidate amplitude screen, 16 kept | `1..6` | phase-free, so it cannot cycle-skip |
| 2 | Adam, 200 steps, all survivors at once | `1..10` | far from the optimum, where a curvature model is a liability |
| 3 | L-BFGS strong-Wolfe, eps 2.0 -> 1.5 -> 1.0 cells | full | locally quadratic; the anneal sharpens the interface as the estimate sharpens |

Stage 1 keeps 16 survivors rather than 1 because the grid spacing is about
`0.3 lambda_s` and the basin is about `lambda_s/4`: the true optimum can fall
between nodes, leaving the nearest few nodes all mediocre and nearly tied, so
picking one would be a coin flip. Stage 3 rebuilds the L-BFGS history at each
`eps` -- the objective changes when `eps` does, and curvature estimated on the
old objective is worse than no curvature at all.

In [ ]:
res = INV.invert(fwd, case0, family=family, log=True)
print()
print(res.summary())
st = res.stages
print(f"\nstage 1 best J {st['stage1_best_J']:.4e}   "
      f"stage 2 J {st['stage2_J']:.4e}   stage 3 J {st['stage3_J']:.4e}")
print(f"stage 2 started from {len(st['stage1_J'])} survivors, "
      f"kept theta {np.round(st['stage2_theta'], 4)}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.0))

j1 = np.asarray(st["stage1_J"])
ax[0].plot(np.arange(len(j1)), j1, "o-", ms=4)
ax[0].set(xlabel="survivor rank", ylabel="amplitude misfit",
         title=f"stage 1: {cfg.SCREEN_GRID**2} candidates,\n"
               f"{cfg.N_SURVIVORS} kept -- note how nearly tied")

ax[1].semilogy(st["stage2_trace"], lw=1.1)
ax[1].set(xlabel="Adam step", ylabel="mean J over survivors",
         title=f"stage 2: band 1..{cfg.BAND_STAGE2.stop}, "
               f"lr {cfg.ADAM_LR_STAGE2}")

tr3 = np.asarray(st["stage3_trace"])
ax[2].semilogy(tr3, lw=1.1)
n_eps = 3
for b in range(1, n_eps):
    ax[2].axvline(b * len(tr3) / n_eps, ls=":", c="0.45", lw=1.0)
ax[2].set(xlabel="closure evaluation", ylabel="J + Tikhonov",
         title=f"stage 3: L-BFGS, eps {cfg.EPS_INVERT_START} -> "
               f"{cfg.EPS_INVERT_END} cells\n(dotted: fresh history per eps)")
fig.tight_layout()
savefig(fig, "05_stage_traces.png")
plt.show()

## Figure 4 -- the misfit landscape

`J` over `(xc, yc)` at fixed `R`, computed twice on the same case:

- **left**, the stage-1 objective: scale-invariant amplitude misfit over the
  lowest 6 lines;
- **right**, the stage-3 objective: complex misfit over the full band.

Two features are being looked for, and both are physics rather than
decoration.

The basin should be about `lambda_s/4` across -- the resolution a
half-wavelength criterion predicts -- and it should be **elongated along the
source-to-defect line**. One source constrains travel time, hence range, far
better than it constrains angle, so a circular basin would mean the misfit is
being driven by amplitude rather than by phase.

And the right panel should show *concentric ripples* that the left panel does
not. Those are cycle skips: move the trial void half a wavelength and the
modelled arrival slips by a full period, so the complex misfit returns almost
to its minimum at a place that is wrong. The amplitude misfit has no carrier in
it and so has no ripples -- which is the entire argument for screening on
amplitude at low frequency before letting phase near the problem.

In [ ]:
obj_screen = Objective(fwd, case0, family, band=cfg.BAND_STAGE1,
                       eps_cells=cfg.EPS_INVERT_START, amplitude=True,
                       scale_invariant=True)
obj_full = Objective(fwd, case0, family, band=cfg.BAND_STAGE3,
                     eps_cells=cfg.EPS_INVERT_END)

t0 = time.perf_counter()
m_screen = misfit_map(obj_screen, n=MAP_N)
m_full = misfit_map(obj_full, n=MAP_N)
print(f"two {MAP_N}x{MAP_N} maps in {time.perf_counter()-t0:.1f} s")

b_screen = basin_width(m_screen)
b_full = basin_width(m_full)
table([("stage 1 (amplitude, band 1..6)", f"{b_screen['along_ls']:.3f}",
        f"{b_screen['across_ls']:.3f}",
        f"{b_screen['along_ls']/max(b_screen['across_ls'],1e-9):.2f}",
        f"{b_screen['angle_deg']:.1f}"),
       ("stage 3 (complex, full band)", f"{b_full['along_ls']:.3f}",
        f"{b_full['across_ls']:.3f}",
        f"{b_full['along_ls']/max(b_full['across_ls'],1e-9):.2f}",
        f"{b_full['angle_deg']:.1f}")],
      ["objective", "along (l_s)", "across (l_s)", "elongation",
       "source angle (deg)"])
print(f"\nlambda_s/4 = {0.25:.2f} lambda_s is the half-wavelength resolution estimate")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.6, 4.2))
for a_, m, ttl in [(ax[0], m_screen, "stage 1: amplitude, band 1..6"),
                   (ax[1], m_full, "stage 3: complex, full band")]:
    J = _np(m["J"])
    x, y = _np(m["x"]), _np(m["y"])
    im = a_.pcolormesh(x, y, np.log10(J), shading="nearest", cmap="viridis")
    a_.contour(x, y, J, levels=[2.0 * J.min()], colors="w", linewidths=1.2)
    tt = _np(m["theta_true"])
    sx_, sy_ = m["source_xy"]
    a_.plot(tt[0], tt[1], "r*", ms=13, label="truth")
    a_.plot(*m["argmin"], "wx", ms=8, mew=2, label="grid argmin")
    a_.plot(_np(res.theta)[0], _np(res.theta)[1], "co", ms=6, mfc="none", mew=1.6,
            label="inverted")
    a_.plot([sx_, tt[0]], [sy_, tt[1]], "-", c="w", lw=0.8, alpha=0.7)
    a_.plot(sx_, sy_, "w^", ms=8, label="source")
    ls = m["lambda_s"]
    a_.plot([x[1], x[1] + 0.25 * ls], [y[1], y[1]], "-", c="w", lw=3)
    a_.text(x[1], y[1] + 0.06 * ls, "lambda_s/4", color="w", fontsize=7.5)
    a_.set(xlabel="xc", ylabel="yc", title=f"{ttl}\nlog10 J at R = {m['radius']:.4f}",
           aspect="equal")
    a_.grid(False)
    a_.legend(fontsize=7, loc="upper right", framealpha=0.7)
    fig.colorbar(im, ax=a_, fraction=0.046)
fig.tight_layout()
savefig(fig, "05_fig4_misfit_landscape.png")
plt.show()

### The same thing as a 1-D slice

Along the source-to-defect line, which is the direction cycle skips live in.
The complex misfit should oscillate with a period near `lambda_s/2` -- half a
wavelength of position moves the two-way path by a full wavelength -- while the
amplitude misfit stays monotone into the basin. Every local minimum in the blue
curve is a place a gradient-based inversion started at the wrong side of would
happily converge to and report a confident answer.

In [ ]:
tt = _np(case0.theta_true)
sx_, sy_ = cfg.SOURCE_XY[case0.src_idx]
ang = math.atan2(tt[1] - sy_, tt[0] - sx_)
s_ = np.linspace(-1.6, 1.6, 161) * case0.lambda_s
th_line = torch.tensor(
    np.stack([tt[0] + s_ * math.cos(ang), tt[1] + s_ * math.sin(ang),
              np.full_like(s_, tt[2])], axis=-1),
    dtype=torch.float32, device=DEV)

with torch.no_grad():
    js, jf = [], []
    for k in range(0, th_line.shape[0], cfg.SCREEN_CHUNK):
        c = th_line[k:k + cfg.SCREEN_CHUNK]
        js.append(_np(obj_screen.residual(c)))
        jf.append(_np(obj_full.residual(c)))
js, jf = np.concatenate(js), np.concatenate(jf)

fig, ax = plt.subplots(figsize=(7.0, 3.2))
ax.semilogy(s_ / case0.lambda_s, jf, lw=1.2, c="C0",
            label="complex, full band (stage 3)")
ax.semilogy(s_ / case0.lambda_s, js, lw=1.2, c="C1",
            label="amplitude, band 1..6 (stage 1)")
loc = [i for i in range(1, len(jf) - 1) if jf[i] < jf[i-1] and jf[i] < jf[i+1]]
ax.plot(s_[loc] / case0.lambda_s, jf[loc], "v", ms=5, c="C3",
        label=f"{len(loc)} local minima in the complex misfit")
ax.axvline(0.0, ls="--", c="0.4", lw=1.0, label="truth")
for k in (-1.0, -0.5, 0.5, 1.0):
    ax.axvline(k * 0.5, ls=":", c="0.75", lw=0.8)
ax.set(xlabel="displacement along the source-defect line (lambda_s)",
       ylabel="J", title="cycle skipping: dotted lines every lambda_s/2")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_cycle_skipping.png")
plt.show()

### What happens if you skip the screen

`skip_screen=True` with a starting guess a wavelength away, which is what a
single-stage gradient inversion amounts to. The screen exists because this is
the alternative -- and note that the failure is not obvious from the outside:
the final misfit is small, the optimiser reports convergence, and the answer is
wrong by more than a wavelength. That is exactly the failure mode the
model-mismatch detector of stage F is meant to catch, and it is why the misfit
statistic is reported alongside every position estimate.

In [ ]:
lo_b, hi_b = family.bounds(case0.lambda_s)
lo_b, hi_b = lo_b.to(DEV), hi_b.to(DEV)
off = 0.75 * case0.lambda_s          # between two cycle-skip minima, not at one
bad = case0.theta_true.clone()
bad[0] += off * math.cos(ang)
bad[1] += off * math.sin(ang)
bad = torch.clamp(bad, lo_b + 1e-3, hi_b - 1e-3)     # bounds are enforced anyway
d_start = float((bad[:2] - case0.theta_true[:2]).norm()) / case0.lambda_s

res_bad = INV.invert(fwd, case0, family=family, theta_init=bad.to(DEV),
                     skip_screen=True)
print(f"start        {np.round(_np(bad), 4)}  ({d_start:.2f} lambda_s from truth, "
      f"along the source-defect line)")
print(f"converged to {np.round(_np(res_bad.theta), 4)}")
print(f"truth        {np.round(_np(case0.theta_true), 4)}")
table([("with the screen", f"{res.position_error_ls:.4f}", f"{res.misfit:.4e}",
        "PASS" if res.success else "FAIL"),
       ("screen skipped, bad start", f"{res_bad.position_error_ls:.4f}",
        f"{res_bad.misfit:.4e}", "PASS" if res_bad.success else "FAIL")],
      ["run", "position error (l_s)", "final misfit", f"gate < {cfg.GATE_POSITION_LS}"])
print(f"\nthe bad run moved {float((res_bad.theta[:2]-bad[:2]).norm())/case0.lambda_s:.3f}"
      f" lambda_s from where it started, and stopped "
      f"{res_bad.position_error_ls:.3f} lambda_s from the answer")

## Step 11 -- the success-rate statistic

`N_CASES` test samples at 30 dB, half on trained illuminations and half on the
held-out sources `SRC_HELDOUT = (3, 6)`, so the generalisation split is balanced
rather than incidental.

Noise is added to the **total** velocity A-scans in the time domain, and the
incident field is subtracted afterwards. That order matters and
`load_inversion_case` enforces it: a real instrument measures the total field,
so the noise floor is set by the total amplitude, which near the source is far
larger than the scattered signal. Noising the residual instead would make the
inversion look good at SNRs where it would in fact fail.

`summarise` reports the median position error next to the mean because the
failure is bimodal rather than heavy-tailed: an inversion either lands in the
right basin (`~lambda_s/20`) or skips into a neighbouring one (`~lambda_s/2`),
and a mean over that describes neither mode.

In [ ]:
rng = np.random.default_rng(cfg.SEED)
idx_held = rng.choice(np.where(held_mask)[0], size=min(N_CASES // 2,
                                                       int(held_mask.sum())),
                      replace=False)
idx_tr = rng.choice(np.where(~held_mask)[0], size=N_CASES - len(idx_held),
                    replace=False)
sel = np.concatenate([idx_tr, idx_held])
is_held = np.concatenate([np.zeros(len(idx_tr), bool), np.ones(len(idx_held), bool)])
print(f"{len(sel)} cases: {len(idx_tr)} trained sources, {len(idx_held)} held out")


def make_cases(indices, snr_db, seed=cfg.SEED):
    out = []
    for k, i in enumerate(indices):
        g = torch.Generator().manual_seed(int(seed) + int(i))
        d = load_inversion_case(str(paths["test"]), int(i), snr_db=snr_db,
                                generator=g)
        out.append(InverseCase.from_dict(d).to(DEV))
    return out


cases30 = make_cases(sel, 30.0)
t0 = time.perf_counter()
res30 = INV.run_many(fwd, cases30, family=family, progress=tqdm)
print(f"\n{len(res30)} inversions in {(time.perf_counter()-t0)/60:.1f} min "
      f"({(time.perf_counter()-t0)/len(res30):.1f} s each)")

s_all = INV.summarise(res30)
s_tr = INV.summarise([r for r, h in zip(res30, is_held) if not h])
s_he = INV.summarise([r for r, h in zip(res30, is_held) if h])
table([(k, f"{s_all[k]}", f"{s_tr[k]}", f"{s_he[k]}") for k in
       ("n", "success_rate", "position_ls_median", "position_ls_mean",
        "position_ls_p90", "radius_ls_median", "misfit_median", "seconds_mean")],
      ["", "all", "trained sources", f"held out {cfg.SRC_HELDOUT}"])
print(f"\ngate: success rate >= {cfg.GATE_SUCCESS_RATE:.0%} at 30 dB  ->  "
      f"{'PASS' if s_all['gate_pass'] else 'FAIL'} "
      f"({s_all['success_rate']:.1%})")

In [ ]:
pos30 = np.array([r.position_error_ls for r in res30])
mis30 = np.array([r.misfit for r in res30])
rad_true = np.array([float(r.theta_true[2]) for r in res30])
lam30 = np.array([r.lambda_s for r in res30])

fig, ax = plt.subplots(1, 3, figsize=(11.4, 3.0))
bins = np.linspace(0, max(pos30.max() * 1.05, cfg.GATE_POSITION_LS * 2), 30)
ax[0].hist(pos30[~is_held], bins=bins, alpha=0.75, label="trained sources")
ax[0].hist(pos30[is_held], bins=bins, alpha=0.75, label=f"held out {cfg.SRC_HELDOUT}")
ax[0].axvline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_POSITION_LS} lambda_s")
ax[0].set(xlabel="position error / lambda_s", ylabel="count",
          title="bimodal: right basin, or a skip")
ax[0].legend(fontsize=7)

ax[1].loglog(mis30, np.maximum(pos30, 1e-4), "o", ms=4, alpha=0.7)
ax[1].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[1].set(xlabel="final misfit", ylabel="position error / lambda_s",
          title="the misfit knows\n(this is the detector statistic)")

ax[2].plot(rad_true / lam30, pos30, "o", ms=4, alpha=0.7)
ax[2].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[2].axvline(cfg.R_MIN_LS, ls=":", c="0.4", lw=1.0, label="R_MIN_LS")
ax[2].set(xlabel="true R / lambda_s", ylabel="position error / lambda_s",
          yscale="log", title="small voids are the hard ones")
ax[2].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "05_success_statistics.png")
plt.show()

r_ = np.corrcoef(np.log(np.maximum(mis30, 1e-30)),
                 np.log(np.maximum(pos30, 1e-6)))[0, 1]
print(f"corr(log misfit, log position error) = {r_:+.3f} -- the basis for using the "
      f"final misfit\nas a self-diagnostic, and in stage F as the mismatch "
      f"detector")

## The SNR sweep

`SNR_DB_SWEEP = (60, 40, 30, 20)`. 60 dB is effectively noiseless and measures
the surrogate's own error floor; 20 dB is where a scattered signal from a small
void near the noise floor stops being recoverable. The interesting number is
not the success rate at any one SNR but where the curve breaks, because that is
what says whether the 30 dB result is comfortably inside the working range or
sitting on a cliff edge.

In [ ]:
sel_s = sel[:N_SNR_CASES]
held_s = is_held[:N_SNR_CASES]
sweep = {}
for snr in cfg.SNR_DB_SWEEP:
    cs = make_cases(sel_s, float(snr))
    rs = INV.run_many(fwd, cs, family=family)
    sweep[float(snr)] = dict(summary=INV.summarise(rs),
                             pos=[r.position_error_ls for r in rs],
                             misfit=[r.misfit for r in rs])
    s = sweep[float(snr)]["summary"]
    print(f"{snr:5.0f} dB   success {s['success_rate']:6.1%}   "
          f"median pos err {s['position_ls_median']:.4f} lambda_s   "
          f"median misfit {s['misfit_median']:.3e}")

In [ ]:
snrs = sorted(sweep, reverse=True)
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].plot(snrs, [sweep[s]["summary"]["success_rate"] for s in snrs], "o-", ms=5)
ax[0].axhline(cfg.GATE_SUCCESS_RATE, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_SUCCESS_RATE:.0%}")
ax[0].axvline(30.0, ls=":", c="0.4", lw=1.0, label="the quoted 30 dB")
ax[0].set(xlabel="SNR (dB)", ylabel="success rate", ylim=(-0.05, 1.05),
          title=f"success vs SNR ({N_SNR_CASES} cases each)")
ax[0].invert_xaxis()
ax[0].legend(fontsize=7.5)

for s in snrs:
    p = np.maximum(np.asarray(sweep[s]["pos"]), 1e-4)
    ax[1].semilogy([s] * len(p), p, "o", ms=4, alpha=0.55)
ax[1].semilogy(snrs, [sweep[s]["summary"]["position_ls_median"] for s in snrs],
               "k-", lw=1.2, label="median")
ax[1].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[1].set(xlabel="SNR (dB)", ylabel="position error / lambda_s",
          title="per-case errors")
ax[1].invert_xaxis()
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "05_snr_sweep.png")
plt.show()

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "quick": QUICK,
    "checkpoint": str(CKPT), "arch": meta["arch"],
    "gradient_check": dict(
        digits=worst_digits, gate=cfg.GATE_GRAD_SIGFIGS,
        passed=bool(worst_digits >= cfg.GATE_GRAD_SIGFIGS),
        best_h_over_lambda_s=best_h / lam_s,
        autodiff=grad_ad.tolist(), fd=best.tolist(),
        offset_lambda_s=float(delta[:2].norm()) / lam_s),
    "single_case": dict(
        index=int(i0), src_idx=int(case0.src_idx),
        held_out=bool(src_all[i0] in cfg.SRC_HELDOUT),
        theta_true=_np(case0.theta_true).tolist(),
        theta=_np(res.theta).tolist(), misfit=res.misfit,
        position_ls=res.position_error_ls, radius_ls=res.radius_error_ls,
        seconds=res.seconds, n_forward=res.n_forward,
        stage1_best_J=st["stage1_best_J"], stage2_J=st["stage2_J"],
        stage3_J=st["stage3_J"]),
    "skip_screen_control": dict(
        theta_start=_np(bad).tolist(), start_offset_ls=d_start,
        theta=_np(res_bad.theta).tolist(), misfit=res_bad.misfit,
        position_ls=res_bad.position_error_ls, success=res_bad.success),
    "basin": dict(screen=b_screen, full=b_full,
                  n_local_minima_along_line=len(loc)),
    "step11_30db": dict(all=s_all, trained=s_tr, heldout=s_he,
                        indices=sel.tolist(), held=is_held.tolist(),
                        position_ls=pos30.tolist(), misfit=mis30.tolist(),
                        corr_logmisfit_logerror=float(r_)),
    "snr_sweep": {str(k): v for k, v in sweep.items()},
}
dump(record, "05_inversion.json")

print()
gates = {
    f"gradient agrees to {cfg.GATE_GRAD_SIGFIGS} sig figs":
        worst_digits >= cfg.GATE_GRAD_SIGFIGS,
    f"single case within {cfg.GATE_POSITION_LS} lambda_s": res.success,
    f"success rate >= {cfg.GATE_SUCCESS_RATE:.0%} at 30 dB": s_all["gate_pass"],
    "held-out illuminations also pass": s_he.get("gate_pass", False),
}
for k, v in gates.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("\nStage F: the RingCNN baseline, the out-of-family transfer, and the\n"
      "model-mismatch detector -- the three results the thesis claim rests on.")

---
# Stage F -- Transfer, the CNN baseline, and the mismatch detector

The last three results, and the ones the thesis claim actually rests on.

- **the baseline** (§8.3). A CNN that regresses `(xc, yc, R)` straight from the
  receiver ring. It is 100x cheaper than the inversion and it is the honest
  comparison: if it matches the four-stage pipeline, the differentiable
  forward model bought nothing.
- **out-of-family transfer** (§8.5, §11.2 step 13). Ellipses and two-void
  geometries, solved with the FDTD, inverted with a surrogate that was trained
  on circles only. Nothing is retrained.
- **the mismatch detector** (§11.2 step 12). The final relative misfit as a test
  statistic for "is this defect the kind of defect I can represent?", scored as
  an ROC.

The out-of-family data comes from fresh solver runs, not from the network. That
is the whole point: the network has to be asked about a shape it has never seen,
and the answer has to be compared against the real physics of that shape.

**Runtime.** The FDTD solves are minutes; the CNN trains in minutes; the
inversions dominate. `QUICK = True` keeps the case counts short.

In [ ]:
N_IN = 12 if QUICK else 24            # in-family cases, the detector's null class
N_ELLIPSE = 6 if QUICK else 12        # out-of-family: ellipses
N_TWO = 6 if QUICK else 12            # out-of-family: two voids
N_SEED = 6 if QUICK else 12           # stage-0 seeding comparison
REG_EPOCHS = 30 if QUICK else 200
SNR = 30.0                            # the SNR every number in this stage is quoted at

print(f"{'QUICK' if QUICK else 'FULL'}: {N_IN} in-family, "
      f"{N_ELLIPSE} ellipse + {N_TWO} two-void out-of-family, at {SNR:.0f} dB")
print(f"RingCNN: {REG_EPOCHS} epochs")

In [ ]:
import csv

import h5py

from src.data import generate as G
from src.data.dataset import load_incident, load_inversion_case
from src.geometry.sdf import (Circle, Ellipse, TwoCircle, fine_coords,
                              geometry_channels, material_fields, soft_indicator)
from src.inverse import invert as INV
from src.inverse.misfit import InverseCase, Objective, SurrogateForward
from src.models import cnn_regressor as CNN
from src.solver import harmonic as H
from src.solver.fdtd_elastic import ElasticFDTD2D

for k in ("train", "val", "test"):
    assert paths[k].exists(), f"missing {paths[k]} -- stage B"
assert CKPT.exists(), "no checkpoint -- stage C"

model, meta = training.load(CKPT, device=DEV)
inc = load_incident(str(paths["test"]), device=DEV)
fwd = SurrogateForward(model, inc, device=DEV)
circle, ellipse, twocircle = Circle(), Ellipse(), TwoCircle()
om = H.omegas_tensor(DEV)

with h5py.File(paths["test"], "r") as f:
    src_test = f["samples/src_idx"][:]
    nu_test = f["samples/nu_idx"][:]
    theta_test = f["samples/theta"][:]
print(f"model {meta['arch']}, epoch {meta['epoch']}, device {DEV}")

## Part 1 -- the baseline that has to be beaten

A circular 1-D CNN over the 32 receivers, `4M + 3 = 83` input channels: real and
imaginary parts of both displacement components at all 20 frequencies, plus
three broadcast conditioning channels (source position and centred Poisson
ratio). It regresses the **unconstrained** `z`, not `theta`, because an MSE on
`theta` would weight the two positions about 40x more heavily than the radius
purely through their units and the radius would never be learned.

The input is normalised by the **receiver-space** incident scale, not the domain
scale. Those differ by roughly two orders of magnitude -- the domain scale is set
by the near-source singularity -- and using the wrong one compresses the whole
input to the bottom of float32's useful range. Stage B measured that ratio; this
is the second place it matters.

Noise is added at the same `SNR` and in the same order (on the total A-scans,
before the incident field is subtracted) as the inversion sees, so the two are
compared on the same data and not on two different problems.

In [ ]:
gtr = torch.Generator().manual_seed(cfg.SEED)
gva = torch.Generator().manual_seed(cfg.SEED + 1)
gte = torch.Generator().manual_seed(cfg.SEED + 2)

t0 = time.perf_counter()
rtr = CNN.ring_features(str(paths["train"]), snr_db=SNR, generator=gtr, device=DEV)
rva = CNN.ring_features(str(paths["val"]), snr_db=SNR, generator=gva, device=DEV)
rte = CNN.ring_features(str(paths["test"]), snr_db=SNR, generator=gte, device=DEV)
print(f"ring features in {time.perf_counter()-t0:.1f} s")
print(f"train {tuple(rtr.x.shape)}  val {tuple(rva.x.shape)}  test {tuple(rte.x.shape)}")
print(f"RING_CHANNELS = {CNN.RING_CHANNELS} = 4 x {cfg.M_FREQ} + 3   "
      f"({rtr.x.numel()*4/1e6:.1f} MB in memory for train)")

In [ ]:
net, hist = CNN.train_regressor(rtr, rva, device=DEV, epochs=REG_EPOCHS,
                                log_every=max(REG_EPOCHS // 8, 1))
CNN.save(net, E.checkpoints / "ringcnn.pt")
print(f"\n{net.n_params():,} parameters "
      f"({net.n_params()/model.effective_params():.3%} of the FNO's)")

held_te = np.isin(src_test, cfg.SRC_HELDOUT)
sc_val = CNN.score(net, rva)
sc_te = CNN.score(net, rte)
sub = lambda m: CNN.RingData(rte.x[m], rte.theta[m], rte.nu[m], rte.src_idx[m])
m_he = torch.from_numpy(held_te).to(rte.x.device)
sc_tr_src = CNN.score(net, sub(~m_he))
sc_he_src = CNN.score(net, sub(m_he))

table([(k, f"{sc_val[k]}", f"{sc_te[k]}", f"{sc_tr_src[k]}", f"{sc_he_src[k]}")
       for k in ("position_ls_mean", "position_ls_median", "radius_ls_mean",
                 "success_rate")],
      ["RingCNN", "val", "test", "test: trained src", f"test: held out {cfg.SRC_HELDOUT}"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].semilogy(hist["train"], lw=1.0, label="train")
ax[0].semilogy(hist["val"], lw=1.0, label="val")
ax[0].set(xlabel="epoch", ylabel="MSE on z", title="RingCNN training")
ax[0].legend(fontsize=8)

with torch.no_grad():
    th_hat = net.predict_theta(rte.x, rte.nu, circle)
lam_te = np.array([cfg.cs_over_cp(float(v)) / cfg.FC for v in _np(rte.nu)])
err_cnn = (_np((th_hat[:, :2] - rte.theta[:, :2]).pow(2).sum(-1).sqrt()) / lam_te)
bins = np.linspace(0, max(err_cnn.max() * 1.02, 0.5), 40)
ax[1].hist(err_cnn[~held_te], bins=bins, alpha=0.75, label="trained sources")
ax[1].hist(err_cnn[held_te], bins=bins, alpha=0.75, label=f"held out {cfg.SRC_HELDOUT}")
ax[1].axvline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_POSITION_LS} lambda_s")
ax[1].set(xlabel="position error / lambda_s", ylabel="count",
          title="RingCNN on the test split")
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_ringcnn.png")
plt.show()

### Does stage 0 actually help?

`invert` *adds* the CNN's guess to the screen's survivors rather than replacing
them, and the asymmetry is deliberate: if the CNN is right the extra candidate
costs one row in a batch of 17, and if the defect is out of distribution --
which is what the rest of this stage is about -- the CNN's guess can be badly
wrong and must not be the only starting point.

The comparison below is the same cases inverted with and without the seed. What
is being looked for is not a better final answer -- both should reach the same
basin on in-family data -- but a cheaper path to it, visible as a lower stage-2
objective from the first step.

In [ ]:
rng = np.random.default_rng(cfg.SEED + 7)
sel_seed = rng.choice(len(theta_test), size=N_SEED, replace=False)


def get_case(i, snr_db=SNR, seed_off=0):
    g = torch.Generator().manual_seed(int(cfg.SEED) + int(i) + int(seed_off))
    return InverseCase.from_dict(
        load_inversion_case(str(paths["test"]), int(i), snr_db=snr_db,
                            generator=g)).to(DEV)


rows, seed_rec = [], []
for i in sel_seed:
    case = get_case(int(i))
    j = int(nu_test[i])
    x = CNN.pack_ring(case.d_obs.cpu(), src_idx=torch.tensor([case.src_idx]),
                      nu=torch.tensor([case.nu]),
                      scale=inc["scale_recv"][case.src_idx, j].cpu().unsqueeze(0))
    with torch.no_grad():
        th0 = net.predict_theta(x.to(DEV), torch.tensor([case.nu], device=DEV),
                                circle)[0]
    r_no = INV.invert(fwd, case, family=circle)
    r_yes = INV.invert(fwd, case, family=circle, theta_init=th0)
    e0 = float((th0[:2].cpu() - case.theta_true[:2].cpu()).norm()) / case.lambda_s
    rows.append((int(i), f"{e0:.3f}", f"{r_no.position_error_ls:.4f}",
                 f"{r_yes.position_error_ls:.4f}",
                 f"{r_no.stages['stage2_trace'][0]:.3e}",
                 f"{r_yes.stages['stage2_trace'][0]:.3e}"))
    seed_rec.append(dict(index=int(i), stage0_error_ls=e0,
                         no_seed=r_no.position_error_ls,
                         seeded=r_yes.position_error_ls,
                         no_seed_misfit=r_no.misfit, seeded_misfit=r_yes.misfit))
table(rows, ["test i", "stage 0 err", "final, no seed", "final, seeded",
             "stage 2 J[0], no seed", "stage 2 J[0], seeded"])
d0 = np.array([r["no_seed"] for r in seed_rec])
d1 = np.array([r["seeded"] for r in seed_rec])
print(f"\nmedian position error  no seed {np.median(d0):.4f}   "
      f"seeded {np.median(d1):.4f} lambda_s")

## Part 2 -- out-of-family geometries, solved properly

Ellipses (aspect ratio 1.5-2.4, random orientation) and pairs of circles
(separation 1.6-3.0 mean radii). Both families are in `sdf.py` and neither
appears anywhere in the training data.

The solver run is set up exactly as `generate.py` sets it up -- same fine grid,
same interface width in *physical* units (`EPS_LEN_PHYS`, the width the network
sees, not that many cells of the finer grid), same source injection, same
receiver sampling from the downsampled field. The only change is which SDF
makes `chi`. Anything else would confound "the network has not seen this shape"
with "the data was made differently".

Sources are drawn from `SRC_TRAIN` on purpose. The point of this experiment is
to vary one thing, and that thing is the shape.

**The equivalent circle.** Scoring a circle fit against a non-circular truth
needs a reference. The convention here is equal area: `R_eq = sqrt(ab)` for an
ellipse, `sqrt(R1^2 + R2^2)` for two voids, with the area-weighted centroid as
the centre. It is a convention and not a ground truth -- there is no correct
circle for an ellipse -- which is exactly why the misfit, and not the position
error, is what the detector is built on.

In [ ]:
def sample_out(n, fam, rng):
    th, ss, jj = [], [], []
    while len(th) < n:
        j = int(rng.integers(len(cfg.NU_LIST)))
        lam = cfg.cs_over_cp(cfg.NU_LIST[j]) / cfg.FC
        s = int(rng.choice(cfg.SRC_TRAIN))
        sx, sy = cfg.SOURCE_XY[s]
        if fam.name == "ellipse":
            r_eq = float(rng.uniform(1.15 * cfg.R_MIN_LS, 0.85 * cfg.R_MAX_LS)) * lam
            ar = float(rng.uniform(1.5, 2.4))
            a, b = r_eq * math.sqrt(ar), r_eq / math.sqrt(ar)
            ext = a
        else:
            r1 = float(rng.uniform(cfg.R_MIN_LS, 0.8 * cfg.R_MAX_LS)) * lam
            r2 = r1 * float(rng.uniform(0.6, 1.0))
            # 1.6-3.0 mean radii apart: overlapping peanut at the low end,
            # two resolved voids at the high end.
            sep = 0.5 * (r1 + r2) * float(rng.uniform(1.6, 3.0))
            ang = float(rng.uniform(0, 2 * math.pi))
        pad = cfg.BOUNDARY_KEEPOUT_LS * lam
        if fam.name == "ellipse":
            keep = ext + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            if math.hypot(xc - sx, yc - sy) < ext + G.SRC_KEEPOUT_LS * lam:
                continue
            al = float(rng.uniform(-math.pi / 2, math.pi / 2))
            th.append([xc, yc, a, b, al])
        else:
            keep = max(r1, r2) + 0.5 * sep + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            dx_, dy_ = 0.5 * sep * math.cos(ang), 0.5 * sep * math.sin(ang)
            c1 = (xc + dx_, yc + dy_)
            c2 = (xc - dx_, yc - dy_)
            ok = all(math.hypot(c[0] - sx, c[1] - sy) > r + G.SRC_KEEPOUT_LS * lam
                     for c, r in ((c1, r1), (c2, r2)))
            ok &= all(pad + r <= v <= cfg.L_DOMAIN - pad - r
                      for c, r in ((c1, r1), (c2, r2)) for v in c)
            if not ok:
                continue
            th.append([c1[0], c1[1], r1, c2[0], c2[1], r2])
        ss.append(s)
        jj.append(j)
    return (np.asarray(th, np.float32), np.asarray(ss, np.int64),
            np.asarray(jj, np.int64))


def equivalent_circle(theta, fam):
    t = np.atleast_2d(np.asarray(theta, np.float64))
    if fam.name == "ellipse":
        return np.stack([t[:, 0], t[:, 1], np.sqrt(t[:, 2] * t[:, 3])], axis=-1)
    a1, a2 = t[:, 2] ** 2, t[:, 5] ** 2
    w = a1 + a2
    return np.stack([(a1 * t[:, 0] + a2 * t[:, 3]) / w,
                     (a1 * t[:, 1] + a2 * t[:, 4]) / w,
                     np.sqrt(w)], axis=-1)


rng = np.random.default_rng(cfg.SEED + 11)
th_e, src_e, nu_e = sample_out(N_ELLIPSE, ellipse, rng)
th_t, src_t, nu_t = sample_out(N_TWO, twocircle, rng)
print(f"ellipses  {th_e.shape}  aspect ratios "
      f"{np.round(th_e[:, 2]/th_e[:, 3], 2)}")
print(f"two-void  {th_t.shape}  radius ratios "
      f"{np.round(th_t[:, 5]/th_t[:, 2], 2)}")

In [ ]:
inc_ascans = inc["ascans"].cpu()
inc_scale_r = inc["scale_recv"].cpu()


def solve_out(theta, fam, src_idx, nu_idx, *, snr_db=SNR, seed=0):
    # FDTD -> scattered displacement phasors at the ring, [B,R,2,M] complex.
    yy_f, xx_f = fine_coords(device=DEV)
    out = []
    for lo in range(0, len(theta), cfg.GEN_BATCH):
        hi = min(lo + cfg.GEN_BATCH, len(theta))
        th = torch.as_tensor(theta[lo:hi], device=DEV)
        chi = soft_indicator(fam.sdf(th, yy_f, xx_f), G.EPS_LEN_PHYS)
        lm = [cfg.lame_from_nu(cfg.NU_LIST[int(j)]) for j in nu_idx[lo:hi]]
        lam0 = torch.tensor([v[0] for v in lm], device=DEV).view(-1, 1, 1)
        mu0 = torch.tensor([v[1] for v in lm], device=DEV).view(-1, 1, 1)
        lam, mu, rho = material_fields(chi, lam0, mu0)
        sim = ElasticFDTD2D(lam, mu, rho)
        src = [cfg.net_to_fine(*cfg.SOURCES_NET[int(s)]) for s in src_idx[lo:hi]]
        res = sim.run(src, nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
        a_tot = res.ascans.cpu()
        a_inc = inc_ascans[torch.as_tensor(src_idx[lo:hi]),
                           torch.as_tensor(nu_idx[lo:hi])]
        if snr_db is not None:
            g = torch.Generator().manual_seed(int(cfg.SEED) + seed + lo)
            a_tot = H.add_measurement_noise(a_tot, snr_db, generator=g)
        d = (H.displacement_from_ascans(a_tot, omegas=om)
             - H.displacement_from_ascans(a_inc, omegas=om))
        out.append(d)
        del sim, chi, lam, mu, rho
    if DEV.startswith("cuda"):
        torch.cuda.empty_cache()
    return torch.cat(out)


t0 = time.perf_counter()
d_e = solve_out(th_e, ellipse, src_e, nu_e, seed=100)
d_t = solve_out(th_t, twocircle, src_t, nu_t, seed=200)
print(f"{len(th_e)+len(th_t)} out-of-family FDTD solves in "
      f"{(time.perf_counter()-t0)/60:.1f} min")
print(f"d_ellipse {tuple(d_e.shape)}   d_two {tuple(d_t.shape)}")

eq_e = equivalent_circle(th_e, ellipse)
eq_t = equivalent_circle(th_t, twocircle)
cases_e = [InverseCase(d_obs=d_e[k:k+1], src_idx=int(src_e[k]),
                       nu_idx=int(nu_e[k]), snr_db=SNR,
                       theta_true=torch.tensor(eq_e[k], dtype=torch.float32)
                       ).to(DEV) for k in range(len(th_e))]
cases_t = [InverseCase(d_obs=d_t[k:k+1], src_idx=int(src_t[k]),
                       nu_idx=int(nu_t[k]), snr_db=SNR,
                       theta_true=torch.tensor(eq_t[k], dtype=torch.float32)
                       ).to(DEV) for k in range(len(th_t))]

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(11.6, 5.8))
for r_, (th_, fam, eq, ttl) in enumerate([(th_e, ellipse, eq_e, "ellipse"),
                                          (th_t, twocircle, eq_t, "two voids")]):
    for c_ in range(3):
        k = c_ % len(th_)
        _, chi = geometry_channels(torch.as_tensor(th_[k:k+1], device=DEV), fam)
        _, chi_eq = geometry_channels(torch.as_tensor(eq[k:k+1], dtype=torch.float32,
                                                      device=DEV), circle)
        gx = np.arange(cfg.N_NET) * cfg.DX_NET
        a_ = ax[r_, c_]
        a_.imshow(_np(chi)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
                  extent=[gx[0], gx[-1], gx[0], gx[-1]])
        a_.contour(gx, gx, _np(chi_eq)[0], levels=[0.5], colors="C3",
                   linewidths=1.2)
        sx, sy = cfg.SOURCE_XY[int((src_e if r_ == 0 else src_t)[k])]
        a_.plot(sx, sy, "C0*", ms=10)
        a_.set(title=f"{ttl} {k}", xticks=[], yticks=[])
        a_.grid(False)
    a_ = ax[r_, 3]
    dd = (d_e if r_ == 0 else d_t)[0]
    im = a_.imshow(_np(dd.abs()[:, 0, :]), origin="lower", aspect="auto",
                   extent=[cfg.FREQS[0], cfg.FREQS[-1], 0, cfg.N_RECV],
                   cmap="magma")
    a_.set(xlabel="f / f_c", ylabel="receiver",
           title=f"|d_obs| x-component\n{ttl} 0, {SNR:.0f} dB")
    a_.grid(False)
    fig.colorbar(im, ax=a_, fraction=0.046)
ax[0, 0].set_ylabel("red: equal-area circle")
fig.tight_layout()
savefig(fig, "06_out_of_family_shapes.png")
plt.show()

## Part 3 -- the detector

Every case is inverted with `Circle()`. The in-family cases are drawn from the
test split at the same SNR and restricted to the same source pool, so the null
class differs from the alternative in the shape and in nothing else.

The statistic is the **final relative misfit**. After convergence, a circle
fitted to a circle's data leaves surrogate error plus measurement noise; a
circle fitted to an ellipse's data leaves structured residual it has no
parameter to absorb. The misfit is normalised by `||d_obs||^2`, which is what
lets a single threshold work across SNRs, source positions and defect sizes --
an absolute residual would need one threshold per acquisition.

Why this matters more than the position error: a localisation tool that silently
returns a confident wrong answer on an unmodelled defect is worse than one that
says it does not know.

In [ ]:
sel_in = rng.choice(np.where(np.isin(src_test, cfg.SRC_TRAIN))[0], size=N_IN,
                    replace=False)
cases_in = [get_case(int(i), seed_off=500) for i in sel_in]

t0 = time.perf_counter()
res_in = INV.run_many(fwd, cases_in, family=circle, progress=tqdm)
res_e = INV.run_many(fwd, cases_e, family=circle, progress=tqdm)
res_t = INV.run_many(fwd, cases_t, family=circle, progress=tqdm)
print(f"\n{len(res_in)+len(res_e)+len(res_t)} inversions in "
      f"{(time.perf_counter()-t0)/60:.1f} min")

mis_in = [r.misfit for r in res_in]
mis_e = [r.misfit for r in res_e]
mis_t = [r.misfit for r in res_t]
roc = INV.detector_roc(mis_in, mis_e + mis_t)
roc_e = INV.detector_roc(mis_in, mis_e)
roc_t = INV.detector_roc(mis_in, mis_t)

table([("circle (in family)", len(mis_in), f"{np.median(mis_in):.4e}",
        f"{INV.summarise(res_in)['position_ls_median']:.4f}", "-"),
       ("ellipse", len(mis_e), f"{np.median(mis_e):.4e}",
        f"{INV.summarise(res_e)['position_ls_median']:.4f}", f"{roc_e['auc']:.3f}"),
       ("two voids", len(mis_t), f"{np.median(mis_t):.4e}",
        f"{INV.summarise(res_t)['position_ls_median']:.4f}", f"{roc_t['auc']:.3f}")],
      ["family inverted as a circle", "n", "median final misfit",
       "median position err (l_s)", "AUC vs in-family"])
print(f"\ncombined AUC = {roc['auc']:.4f}   "
      f"median misfit {roc['median_in']:.3e} (in) vs {roc['median_out']:.3e} (out), "
      f"a factor of {roc['median_out']/max(roc['median_in'],1e-30):.1f}")

In [ ]:
tpr = np.asarray(roc["tpr"])
fpr = np.asarray(roc["fpr"])
thr = np.asarray(roc["threshold"])
k_best = int(np.argmax(tpr - fpr))

fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].plot(fpr, tpr, "-", lw=1.4, label=f"all out-of-family, AUC {roc['auc']:.3f}")
ax[0].plot(roc_e["fpr"], roc_e["tpr"], "--", lw=1.0,
           label=f"ellipse only, AUC {roc_e['auc']:.3f}")
ax[0].plot(roc_t["fpr"], roc_t["tpr"], ":", lw=1.2,
           label=f"two voids only, AUC {roc_t['auc']:.3f}")
ax[0].plot([0, 1], [0, 1], c="0.6", lw=0.8)
ax[0].plot(fpr[k_best], tpr[k_best], "ko", ms=6,
           label=f"Youden: TPR {tpr[k_best]:.2f} at FPR {fpr[k_best]:.2f}")
ax[0].set(xlabel="false positive rate (in-family flagged)",
          ylabel="true positive rate (out-of-family flagged)",
          title="figure 5: model-mismatch detector", aspect="equal")
ax[0].legend(fontsize=7)

for i, (v, lab) in enumerate([(mis_in, "circle"), (mis_e, "ellipse"),
                              (mis_t, "two voids")]):
    j = np.full(len(v), i, float) + rng.normal(0, 0.06, len(v))
    ax[1].semilogy(j, v, "o", ms=5, alpha=0.7)
    ax[1].semilogy([i - 0.25, i + 0.25], [np.median(v)] * 2, "k-", lw=1.6)
ax[1].axhline(thr[k_best], ls="--", c="C3", lw=1.0,
              label=f"threshold {thr[k_best]:.3e}")
ax[1].set(xticks=[0, 1, 2], xticklabels=["circle", "ellipse", "two voids"],
          ylabel="final relative misfit", title="the statistic itself")
ax[1].legend(fontsize=7.5)

pos_in = [r.position_error_ls for r in res_in]
pos_out = [r.position_error_ls for r in res_e + res_t]
ax[2].loglog(mis_in, np.maximum(pos_in, 1e-4), "o", ms=5, alpha=0.75,
             label="in family")
ax[2].loglog(mis_e + mis_t, np.maximum(pos_out, 1e-4), "s", ms=5, alpha=0.75,
             label="out of family (vs equal-area circle)")
ax[2].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[2].axvline(thr[k_best], ls="--", c="0.4", lw=1.0)
ax[2].set(xlabel="final misfit", ylabel="position error / lambda_s",
          title="it still localises --\nit just knows it does not fit")
ax[2].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "06_fig5_detector_roc.png")
plt.show()

## Part 4 -- the transfer, with no retraining

The surrogate was trained on circles. It is now asked about ellipses, and the
*only* thing that changes is which `ShapeFamily` builds the two geometry input
channels. The weights are frozen, the band is the same, the optimiser is the
same.

Stage 1 is skipped, and not for convenience: `screen` and `misfit_map` build
their candidate grid with a three-column `torch.stack`, so the screen is
structurally circle-only. The ellipse run is seeded from the converged circle
fit -- `(xc, yc, R) -> (xc, yc, a=R, b=R, alpha=0)`, a circle expressed in
ellipse coordinates -- and refined from there. This is the honest version of
the §8.5 claim: the transfer works because the network learned an operator on
`(phi_tilde, chi)` fields rather than a map on three numbers, and the evidence
is that releasing two extra degrees of freedom *lowers the misfit* on data the
network has never seen the shape of.

In [ ]:
res_tr, rows = [], []
for k, (ce, rc) in enumerate(zip(cases_e, res_e)):
    x0, y0, r0 = (float(v) for v in rc.theta[:3])
    seed = torch.tensor([x0, y0, r0, r0, 0.0], dtype=torch.float32, device=DEV)
    ce5 = InverseCase(d_obs=ce.d_obs, src_idx=ce.src_idx, nu_idx=ce.nu_idx,
                      snr_db=ce.snr_db,
                      theta_true=torch.tensor(th_e[k], dtype=torch.float32)
                      ).to(DEV)
    r = INV.invert(fwd, ce5, family=ellipse, theta_init=seed, skip_screen=True)
    res_tr.append(r)
    tt, th_hat = th_e[k], _np(r.theta)
    ar_true, ar_hat = tt[2] / tt[3], th_hat[2] / max(th_hat[3], 1e-9)
    da = abs(((th_hat[4] - tt[4] + math.pi / 2) % math.pi) - math.pi / 2)
    rows.append((k, f"{ar_true:.2f}", f"{ar_hat:.2f}",
                 f"{math.degrees(da):.1f}", f"{r.position_error_ls:.4f}",
                 f"{rc.misfit:.3e}", f"{r.misfit:.3e}",
                 f"{100*(1 - r.misfit/max(rc.misfit,1e-30)):+.0f}%"))
table(rows, ["case", "true a/b", "fitted a/b", "orientation err (deg)",
             "position err (l_s)", "misfit as circle", "misfit as ellipse",
             "change"])

drop = np.array([1 - r.misfit / max(c.misfit, 1e-30)
                 for r, c in zip(res_tr, res_e)])
pos_tr = np.array([r.position_error_ls for r in res_tr])
print(f"\nmedian misfit reduction from releasing (b, alpha): {np.median(drop):+.1%}")
print(f"median position error, ellipse family: {np.median(pos_tr):.4f} lambda_s "
      f"(gate {cfg.GATE_POSITION_LS})")
print(f"improved in {int((drop > 0).sum())}/{len(drop)} cases")

In [ ]:
n_show = min(3, len(res_tr))
fig, ax = plt.subplots(1, n_show + 1, figsize=(3.0 * (n_show + 1), 3.1))
gx = np.arange(cfg.N_NET) * cfg.DX_NET
for k in range(n_show):
    a_ = ax[k]
    _, chi_true = geometry_channels(torch.as_tensor(th_e[k:k+1], device=DEV), ellipse)
    _, chi_c = geometry_channels(res_e[k].theta.unsqueeze(0).to(DEV), circle)
    _, chi_t = geometry_channels(res_tr[k].theta.unsqueeze(0).to(DEV), ellipse)
    a_.imshow(_np(chi_true)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
              extent=[gx[0], gx[-1], gx[0], gx[-1]])
    a_.contour(gx, gx, _np(chi_c)[0], levels=[0.5], colors="C3", linewidths=1.3)
    a_.contour(gx, gx, _np(chi_t)[0], levels=[0.5], colors="C2", linewidths=1.3)
    sx, sy = cfg.SOURCE_XY[int(src_e[k])]
    a_.plot(sx, sy, "C0*", ms=10)
    a_.set(title=f"ellipse {k}: grey truth,\nred circle fit, green ellipse fit",
           xticks=[], yticks=[])
    a_.grid(False)

a_ = ax[n_show]
a_.semilogy([0] * len(res_e), [r.misfit for r in res_e], "o", ms=5, alpha=0.7)
a_.semilogy([1] * len(res_tr), [r.misfit for r in res_tr], "s", ms=5, alpha=0.7)
for rc, rt in zip(res_e, res_tr):
    a_.semilogy([0, 1], [rc.misfit, rt.misfit], "-", c="0.6", lw=0.7)
a_.semilogy([-0.2, 0.2], [np.median(mis_e)] * 2, "k-", lw=1.6)
a_.semilogy([0.8, 1.2], [np.median([r.misfit for r in res_tr])] * 2, "k-", lw=1.6)
a_.axhline(np.median(mis_in), ls="--", c="C3", lw=1.0, label="in-family median")
a_.set(xticks=[0, 1], xticklabels=["circle\nfamily", "ellipse\nfamily"],
       ylabel="final misfit", title="no retraining, two extra\ndegrees of freedom")
a_.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_transfer_ellipse.png")
plt.show()

### The baseline on the same data

The RingCNN has three output numbers and no notion of an ellipse, so the most
it can do is report the equal-area circle. It is given exactly the data the
inversion was given -- the same phasors, normalised by the same receiver-space
scale -- and scored against the same convention.

This is the comparison the method has to win, and the reason it wins is
structural rather than about capacity: the regressor learned a map from ring
data to three numbers on a distribution of circles, and an ellipse is off that
distribution with no mechanism to notice. The inversion carries a forward
model, so it can be handed a different family and a different parameter count
at inference time, and it reports a misfit that says when it is out of its
depth.

In [ ]:
def cnn_on(d, src_idx, nu_idx, theta_ref):
    s = torch.as_tensor(src_idx)
    j = torch.as_tensor(nu_idx)
    nu = torch.tensor([cfg.NU_LIST[int(k)] for k in j], dtype=torch.float32)
    x = CNN.pack_ring(d.cpu(), src_idx=s, nu=nu, scale=inc_scale_r[s, j])
    rd = CNN.RingData(x, torch.as_tensor(theta_ref, dtype=torch.float32), nu, s)
    return CNN.score(net, rd.to(DEV))


cnn_e = cnn_on(d_e, src_e, nu_e, eq_e)
cnn_t = cnn_on(d_t, src_t, nu_t, eq_t)
inv_e = INV.summarise(res_e)
inv_t = INV.summarise(res_t)

table([("circle test split", f"{sc_te['position_ls_median']:.4f}",
        f"{INV.summarise(res_in)['position_ls_median']:.4f}"),
       ("ellipse (vs equal-area circle)", f"{cnn_e['position_ls_median']:.4f}",
        f"{inv_e['position_ls_median']:.4f}"),
       ("two voids (vs equal-area circle)", f"{cnn_t['position_ls_median']:.4f}",
        f"{inv_t['position_ls_median']:.4f}")],
      ["median position error (lambda_s)", "RingCNN (stage 0)",
       "four-stage inversion"])
print(f"\nRingCNN: {net.n_params():,} parameters, one forward pass per case.")
print(f"Inversion: {int(np.median([r.n_forward for r in res_in]))} surrogate "
      f"evaluations, {np.median([r.seconds for r in res_in]):.1f} s per case.")
print("The baseline is cheap and it is not wrong -- it is just unable to say when it "
      "is.")

## Part 5 -- the thesis table

Every headline number, with the stage that produced it and the gate it is
measured against. Read from the JSON records in `E.results`, so this cell
reports what was actually run rather than what was intended; anything missing
shows as `-` instead of silently defaulting.

In [ ]:
def load_rec(name):
    p = E.results / name
    return json.loads(p.read_text()) if p.exists() else {}


r1 = load_rec("01_solver_validation.json")
r2 = load_rec("02_dataset_generation.json")
r3 = load_rec("03_train_fno.json")
r4 = load_rec("04_forward_eval.json")
r5 = load_rec("05_inversion.json")


def pick(d, *path, default=None):
    for k in path:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def fmt(v, spec=".4g"):
    return "-" if v is None else format(v, spec) if isinstance(v, float) else str(v)


ck = pick(r1, "checks", default={}) or {}


def check(substr, field="value"):
    for k, v in ck.items():
        if substr in k.lower():
            return v.get(field)
    return None


rows = [
    ("solver checks passed", f"{pick(r1,'n_pass',default='-')} of "
     f"{pick(r1,'n_total',default='-')}", "5 of 5", "A"),
    ("deconvolution amplification",
     fmt(pick(r1, "deconvolution", "worst_amplification")), "< 25", "A"),
    ("grid convergence, 256 vs 512", fmt(check("grid")),
     f"< {cfg.GATE_GRID_CONVERGENCE}", "A"),
    ("absorber residual", fmt(check("absorber"), ".2e"),
     f"< {cfg.GATE_PML_RESIDUAL}", "A"),
    ("dataset re-solve spot check", fmt(pick(r2, "spot_check", "max"), ".2e"),
     f"< {pick(r2,'spot_check','gate',default='-')}", "B"),
    ("wrap-around tail energy",
     fmt(pick(r2, "tail_energy_fraction", "max"), ".2e"), "< 1e-3", "B"),
    ("surrogate field rel-L2 (test)", fmt(pick(r4, "test", "rel_l2")),
     f"< {cfg.GATE_REL_L2}", "D"),
    ("surrogate arrival error", fmt(pick(r4, "test", "phase")),
     f"< {cfg.GATE_ARRIVAL_PERIODS} periods", "D"),
    ("held-out-source penalty",
     fmt(pick(r4, "per_sample", "heldout_penalty")), "reported", "D"),
    ("physics-loss ablation, rel-L2",
     f"{fmt(pick(r3,'arms','nophys','rel_l2'))} -> "
     f"{fmt(pick(r3,'arms','full','rel_l2'))}", "physics <= none", "C"),
    ("L_phys floor (true field)", fmt(pick(r3, "phys_floor_true_field"), ".4e"),
     "reported", "C"),
    ("gradient check, significant figures",
     fmt(pick(r5, "gradient_check", "digits"), ".2f"),
     f">= {cfg.GATE_GRAD_SIGFIGS}", "E"),
    ("inversion success rate at 30 dB",
     fmt(pick(r5, "step11_30db", "all", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "E"),
    ("  on held-out illuminations",
     fmt(pick(r5, "step11_30db", "heldout", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "E"),
    ("basin width along/across",
     f"{fmt(pick(r5,'basin','full','along_ls'),'.3f')} / "
     f"{fmt(pick(r5,'basin','full','across_ls'),'.3f')} lambda_s",
     "~0.25, elongated", "E"),
    ("RingCNN baseline, median position",
     f"{sc_te['position_ls_median']:.4f} lambda_s", "for comparison", "F"),
    ("inversion, median position (in family)",
     f"{INV.summarise(res_in)['position_ls_median']:.4f} lambda_s",
     f"< {cfg.GATE_POSITION_LS}", "F"),
    ("mismatch detector AUC", f"{roc['auc']:.4f}", "> 0.9 desirable", "F"),
    ("ellipse transfer, median position",
     f"{np.median(pos_tr):.4f} lambda_s", f"< {cfg.GATE_POSITION_LS}", "F"),
    ("ellipse transfer, misfit change",
     f"{np.median(drop):+.1%}", "negative is a failure", "F"),
]
table(rows, ["quantity", "value", "gate / expectation", "stage"])

In [ ]:
csv_path = E.results / "thesis_table.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["quantity", "value", "gate", "stage"])
    w.writerows(rows)
print(f"wrote {csv_path}")

record = {
    "device": DEV, "gpu": E.gpu_name, "quick": QUICK, "snr_db": SNR,
    "ringcnn": dict(params=net.n_params(), epochs=REG_EPOCHS,
                    val=sc_val, test=sc_te, test_trained_src=sc_tr_src,
                    test_heldout_src=sc_he_src,
                    on_ellipse=cnn_e, on_two_circle=cnn_t,
                    history=hist),
    "stage0_seeding": seed_rec,
    "out_of_family": dict(
        ellipse=dict(theta=th_e.tolist(), src=src_e.tolist(), nu=nu_e.tolist(),
                     equivalent_circle=eq_e.tolist(), misfit=mis_e,
                     summary=inv_e),
        two_circle=dict(theta=th_t.tolist(), src=src_t.tolist(), nu=nu_t.tolist(),
                        equivalent_circle=eq_t.tolist(), misfit=mis_t,
                        summary=inv_t)),
    "in_family": dict(indices=sel_in.tolist(), misfit=mis_in,
                      summary=INV.summarise(res_in)),
    "detector": dict(auc=roc["auc"], auc_ellipse=roc_e["auc"],
                     auc_two_circle=roc_t["auc"],
                     median_in=roc["median_in"], median_out=roc["median_out"],
                     youden_threshold=float(thr[k_best]),
                     youden_tpr=float(tpr[k_best]), youden_fpr=float(fpr[k_best]),
                     tpr=roc["tpr"], fpr=roc["fpr"],
                     threshold=roc["threshold"]),
    "transfer_ellipse": dict(
        theta=[_np(r.theta).tolist() for r in res_tr],
        misfit_as_circle=mis_e, misfit_as_ellipse=[r.misfit for r in res_tr],
        misfit_reduction=drop.tolist(), position_ls=pos_tr.tolist(),
        median_reduction=float(np.median(drop)),
        median_position_ls=float(np.median(pos_tr))),
    "thesis_table": [list(r) for r in rows],
}
dump(record, "06_transfer_and_detector.json")

## The never-cut checklist

§11.3, in order. These are the things that stay in even when time runs out,
because each one is the only place a particular kind of silent wrongness can be
caught. A `FAIL` or a `-` below is not a note in the discussion section; it is a
number that should not be quoted.

In [ ]:
checks = [
    ("solver sanity checks 1-5",
     (pick(r1, "n_pass") == pick(r1, "n_total") and pick(r1, "include_slow"))
     if r1 else None,
     "an unvalidated solver makes every downstream metric a report on the wrong "
     "physics"),
    ("dataset re-solve spot check", pick(r2, "spot_check", "passed"),
     "the file on disk is what the solver produced"),
    ("forward rel-L2 and arrival gates",
     all(pick(r4, "gates", default={}).values()) if pick(r4, "gates") else None,
     "the surrogate is the operator being inverted; its error is a floor"),
    ("held-out-source generalisation reported",
     pick(r4, "per_sample", "heldout_penalty") is not None,
     "the honest split -- SRC_HELDOUT appears in test and nowhere else"),
    ("gradient check to 3 significant figures",
     pick(r5, "gradient_check", "passed"),
     "a wrong gradient still converges, to the wrong answer"),
    ("misfit landscape figure",
     pick(r5, "basin", "full", "along_ls") is not None,
     "the basin width and its elongation are the resolution claim"),
    ("success rate at 30 dB",
     pick(r5, "step11_30db", "all", "gate_pass"),
     "the headline inverse result"),
    ("SNR sweep", bool(pick(r5, "snr_sweep")),
     "says whether 30 dB is inside the working range or on a cliff edge"),
    ("out-of-family transfer, no retraining",
     bool(np.median(drop) > 0), "the §8.5 claim, and the reason for a forward model"),
    ("model-mismatch detector",
     bool(roc["auc"] > 0.9),
     "a confident wrong answer on an unmodelled defect is the worst failure mode"),
    ("CNN baseline for comparison", bool(sc_te), "otherwise there is no claim"),
]
for name, v, why in checks:
    mark = "----" if v is None else ("PASS" if v else "FAIL")
    print(f"  {mark}  {name}")
    print(f"        {why}")

n_ok = sum(1 for _, v, _ in checks if v)
print(f"\n{n_ok} / {len(checks)}")
if n_ok == len(checks):
    print(f"\nEvery gate in §11.3 is met.  The figures in {E.figures}\n"
          f"and the records in {E.results} are the complete experimental record.\n"
          f"Pull them off the Volume:\n"
          f"  modal volume get fno-wave-inverse-data figures ./figures\n"
          f"  modal volume get fno-wave-inverse-data results ./results")
else:
    print("\nSome gates are unmet or unrun.  The list above says which stage "
          "owns each one;\nrun that stage rather than quoting around the gap.")

## What is left out, on purpose

- **No mixed precision anywhere.** The spectral weights are complex and the
  gradient check of stage E needs double precision; TF32 is off for the same
  reason. The speedup was not worth an unexplained loss of three digits.
- **The screen is circle-only.** `screen` and `misfit_map` stack three columns,
  so an out-of-family inversion has to be seeded. That is a real limitation and
  it is stated rather than papered over -- extending the screen to five
  parameters would be a `16^5` grid, which is the actual reason it was not done.
- **`Ellipse` and `TwoCircle` are never trained on.** They exist to be
  transferred to. Adding them to the training distribution would make the §8.5
  result vacuous.
- **The equal-area circle is a convention.** There is no correct circle for an
  ellipse, which is why the detector is built on the misfit and not on the
  position error.
- **This session trains one arm at a fitted epoch count.** The 300-epoch
  schedule and the `nophys` arm both exist headless
  (`modal run modal_app.py::train_both`); the thesis table reads whatever is on
  the Volume when it runs.